# Orbit Wars JAX PPO — Kaggle GPU

100% self-contained scratch PPO with JAX env + Flax policy.
**Before running:** enable GPU Accelerator (T4x2 or P100) and Internet.


## Setup


In [ ]:
%%capture
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'jax[cuda12]', 'flax', 'optax', 'pyyaml', 'numba'])


In [ ]:
import os
os.makedirs('orbit_wars', exist_ok=True)
os.makedirs('configs', exist_ok=True)


## Environment and Policy source code


In [ ]:
%%writefile orbit_wars/__init__.py

from .constants import *
from .convert import observation_to_state, state_to_observation_dict, states_equal
from .decode import (
    BUCKET_COUNT,
    bucket_validity_mask,
    compose_action_grid,
    launch_angle,
    pack_action_row,
    path_crosses_sun,
    ship_counts_for_buckets,
)
from .env import OrbitWarsJaxEnv, VectorOrbitWarsEnv
from .features_jax import (
    FLEET_FEATURE_DIM,
    GLOBAL_FEATURE_DIM,
    PLANET_FEATURE_DIM,
    encode_batch,
    encode_batch_jit,
    encode_observation,
    encode_observation_jit,
)
from .geometry import distance_xy, fleet_speed, point_to_segment_distance, swept_pair_hit
from .reference import reference_reset, reference_step
from .reset import reset
from .state import OrbitWarsState
from .step import batched_step, step, step_jit

__all__ = [
    "OrbitWarsJaxEnv",
    "VectorOrbitWarsEnv",
    "OrbitWarsState",
    "reset",
    "step",
    "step_jit",
    "batched_step",
    "reference_reset",
    "reference_step",
    "observation_to_state",
    "state_to_observation_dict",
    "states_equal",
    "encode_observation",
    "encode_observation_jit",
    "encode_batch",
    "encode_batch_jit",
    "PLANET_FEATURE_DIM",
    "FLEET_FEATURE_DIM",
    "GLOBAL_FEATURE_DIM",
    "BUCKET_COUNT",
    "compose_action_grid",
    "ship_counts_for_buckets",
    "bucket_validity_mask",
    "path_crosses_sun",
    "launch_angle",
    "pack_action_row",
]


In [ ]:
%%writefile orbit_wars/comet.py
"""Comet spawn logic (vendored from official orbit_wars.py — no kaggle import needed)."""

from __future__ import annotations

import math
import random
from typing import Any

import numpy as np

from .constants import (
    BOARD_SIZE,
    CENTER,
    COMET_PRODUCTION,
    COMET_RADIUS,
    MAX_COMET_PLANETS,
    MAX_PLANETS,
    ROTATION_RADIUS_LIMIT,
    SUN_RADIUS,
)


def _distance(p1: tuple[float, float], p2: tuple[float, float]) -> float:
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def generate_comet_paths(
    initial_planets: list[list[float]],
    angular_velocity: float,
    spawn_step: int,
    comet_planet_ids: list[int] | set[int] | None = None,
    comet_speed: float = 4.0,
    rng: random.Random | None = None,
) -> list[list[list[float]]] | None:
    """Generate 4 symmetric elliptical comet paths (matches official env)."""
    if rng is None:
        rng = random.Random()
    comet_ids = set(comet_planet_ids or [])

    for _ in range(300):
        e = rng.uniform(0.75, 0.93)
        a = rng.uniform(60, 150)
        perihelion = a * (1 - e)
        if perihelion < SUN_RADIUS + COMET_RADIUS:
            continue

        b = a * math.sqrt(1 - e**2)
        c_val = a * e
        phi = rng.uniform(math.pi / 6, math.pi / 3)

        dense: list[tuple[float, float]] = []
        num = 5000
        for i in range(num):
            t = 0.3 * math.pi + 1.4 * math.pi * i / (num - 1)
            ex = c_val + a * math.cos(t)
            ey = b * math.sin(t)
            x = CENTER + ex * math.cos(phi) - ey * math.sin(phi)
            y = CENTER + ex * math.sin(phi) + ey * math.cos(phi)
            dense.append((x, y))

        path = [dense[0]]
        cum = 0.0
        target = comet_speed
        for i in range(1, len(dense)):
            cum += _distance(dense[i], dense[i - 1])
            if cum >= target:
                path.append(dense[i])
                target += comet_speed

        board_start = None
        board_end = None
        for i, (x, y) in enumerate(path):
            if 0 <= x <= BOARD_SIZE and 0 <= y <= BOARD_SIZE:
                if board_start is None:
                    board_start = i
                board_end = i

        if board_start is None:
            continue
        visible = path[board_start : board_end + 1]
        if not (5 <= len(visible) <= 40):
            continue

        paths = [
            [[y, x] for x, y in visible],
            [[BOARD_SIZE - x, y] for x, y in visible],
            [[x, BOARD_SIZE - y] for x, y in visible],
            [[BOARD_SIZE - y, BOARD_SIZE - x] for x, y in visible],
        ]

        static_planets: list[list[float]] = []
        orbiting_planets: list[list[float]] = []
        for planet in initial_planets:
            if planet[0] in comet_ids:
                continue
            pr = _distance((planet[2], planet[3]), (CENTER, CENTER))
            if pr + planet[4] < ROTATION_RADIUS_LIMIT:
                orbiting_planets.append(planet)
            else:
                static_planets.append(planet)

        valid = True
        buf = COMET_RADIUS + 0.5
        for k, (cx, cy) in enumerate(visible):
            if _distance((cx, cy), (CENTER, CENTER)) < SUN_RADIUS + COMET_RADIUS:
                valid = False
                break

            sym_pts = [
                (cy, cx),
                (BOARD_SIZE - cx, cy),
                (cx, BOARD_SIZE - cy),
                (BOARD_SIZE - cy, BOARD_SIZE - cx),
            ]
            for planet in static_planets:
                for sp in sym_pts:
                    if _distance(sp, (planet[2], planet[3])) < planet[4] + buf:
                        valid = False
                        break
                if not valid:
                    break
            if not valid:
                break

            game_step = spawn_step - 1 + k
            for planet in orbiting_planets:
                dx = planet[2] - CENTER
                dy = planet[3] - CENTER
                orb_r = math.sqrt(dx**2 + dy**2)
                init_angle = math.atan2(dy, dx)
                cur_angle = init_angle + angular_velocity * game_step
                px = CENTER + orb_r * math.cos(cur_angle)
                py = CENTER + orb_r * math.sin(cur_angle)
                for sp in sym_pts:
                    if _distance(sp, (px, py)) < planet[4] + COMET_RADIUS:
                        valid = False
                        break
                if not valid:
                    break
            if not valid:
                break

        if valid:
            return paths
    return None


def spawn_comet_for_state(
    planets: np.ndarray,
    n_planets: int,
    initial_planets: np.ndarray,
    comet_planet_ids: np.ndarray,
    n_comet_ids: int,
    angular_velocity: float,
    spawn_step: int,
    episode_seed: int,
    comet_speed: float = 4.0,
) -> dict[str, Any] | None:
    """Return dict with new planet rows + comet group, or None if spawn fails."""
    planet_rows = [
        [float(planets[i, j]) for j in range(7)]
        for i in range(int(n_planets))
        if planets[i, 7] > 0.0
    ]
    initial_rows = [
        [float(initial_planets[i, j]) for j in range(7)]
        for i in range(int(n_planets))
        if initial_planets[i, 7] > 0.0
    ]
    comet_ids = [int(comet_planet_ids[i]) for i in range(int(n_comet_ids)) if int(comet_planet_ids[i]) >= 0]
    comet_rng = random.Random(f"orbit_wars-comet-{episode_seed}-{spawn_step}")
    paths = generate_comet_paths(
        initial_rows,
        float(angular_velocity),
        int(spawn_step),
        comet_ids,
        float(comet_speed),
        rng=comet_rng,
    )
    if not paths:
        return None

    next_id = max(int(p[0]) for p in planet_rows) + 1
    comet_ships = min(
        comet_rng.randint(1, 99),
        comet_rng.randint(1, 99),
        comet_rng.randint(1, 99),
        comet_rng.randint(1, 99),
    )
    new_planets: list[list[float]] = []
    group_pids: list[int] = []
    for i, _path in enumerate(paths):
        pid = next_id + i
        group_pids.append(pid)
        new_planets.append([pid, -1, -99.0, -99.0, COMET_RADIUS, comet_ships, COMET_PRODUCTION])
    return {
        "new_planets": new_planets,
        "group": {"planet_ids": group_pids, "paths": paths, "path_index": -1},
        "new_comet_ids": group_pids,
    }


In [ ]:
%%writefile orbit_wars/constants.py
"""Orbit Wars simulation constants (match official Kaggle env)."""

from __future__ import annotations

BOARD_SIZE = 100.0
CENTER = BOARD_SIZE / 2.0
SUN_RADIUS = 10.0
ROTATION_RADIUS_LIMIT = 50.0
COMET_RADIUS = 1.0
COMET_PRODUCTION = 1
PLANET_CLEARANCE = 7
MIN_PLANET_GROUPS = 5
MAX_PLANET_GROUPS = 10
MIN_STATIC_GROUPS = 3
COMET_SPAWN_STEPS = (50, 150, 250, 350, 450)
DEFAULT_SHIP_SPEED = 6.0
DEFAULT_EPISODE_STEPS = 500

# Padded simulation limits for JIT-friendly arrays.
MAX_PLANETS = 96
MAX_FLEETS = 256
MAX_COMET_GROUPS = 8
MAX_COMET_PATH_LEN = 64
MAX_COMET_PLANETS = MAX_COMET_GROUPS * 4
MAX_MOVES_PER_PLAYER = 48
NUM_PLAYERS = 2

PLANET_COLS = 8  # id, owner, x, y, radius, ships, production, active
FLEET_COLS = 8  # id, owner, x, y, angle, from_planet_id, ships, active


In [ ]:
%%writefile orbit_wars/convert.py
"""Convert between Python list observations and padded JAX state."""

from __future__ import annotations

from typing import Any

import jax.numpy as jnp
import numpy as np

from .constants import (
    FLEET_COLS,
    MAX_COMET_GROUPS,
    MAX_COMET_PATH_LEN,
    MAX_COMET_PLANETS,
    MAX_FLEETS,
    MAX_PLANETS,
    NUM_PLAYERS,
    PLANET_COLS,
)
from .state import CometGroups, OrbitWarsState, empty_comet_groups, empty_state


def _get(obs: Any, key: str, default: Any = None) -> Any:
    if isinstance(obs, dict):
        return obs.get(key, default)
    return getattr(obs, key, default)


def _pack_planets(rows: list[list[float]], *, pad: np.ndarray) -> tuple[np.ndarray, int]:
    out = pad.copy()
    n = min(len(rows), MAX_PLANETS)
    for i, row in enumerate(rows[:n]):
        out[i, 0] = float(row[0])
        out[i, 1] = float(row[1])
        out[i, 2] = float(row[2])
        out[i, 3] = float(row[3])
        out[i, 4] = float(row[4])
        out[i, 5] = float(row[5])
        out[i, 6] = float(row[6])
        out[i, 7] = 1.0
    return out, n


def _pack_fleets(rows: list[list[float]], *, pad: np.ndarray) -> tuple[np.ndarray, int]:
    out = pad.copy()
    n = min(len(rows), MAX_FLEETS)
    for i, row in enumerate(rows[:n]):
        out[i, 0] = float(row[0])
        out[i, 1] = float(row[1])
        out[i, 2] = float(row[2])
        out[i, 3] = float(row[3])
        out[i, 4] = float(row[4])
        out[i, 5] = float(row[5])
        out[i, 6] = float(row[6])
        out[i, 7] = 1.0
    return out, n


def pack_comets(comets: list[dict[str, Any]]) -> CometGroups:
    active = np.zeros((MAX_COMET_GROUPS,), dtype=np.bool_)
    planet_ids = np.full((MAX_COMET_GROUPS, 4), -1, dtype=np.int32)
    path_index = np.full((MAX_COMET_GROUPS,), -1, dtype=np.int32)
    paths = np.zeros((MAX_COMET_GROUPS, 4, MAX_COMET_PATH_LEN, 2), dtype=np.float32)
    path_lengths = np.zeros((MAX_COMET_GROUPS, 4), dtype=np.int32)

    for gi, group in enumerate(comets[:MAX_COMET_GROUPS]):
        active[gi] = True
        path_index[gi] = int(group.get("path_index", -1))
        pids = list(group.get("planet_ids") or [])
        group_paths = list(group.get("paths") or [])
        for pi in range(min(4, len(pids))):
            planet_ids[gi, pi] = int(pids[pi])
            if pi < len(group_paths):
                path = group_paths[pi]
                plen = min(len(path), MAX_COMET_PATH_LEN)
                path_lengths[gi, pi] = plen
                for ti in range(plen):
                    paths[gi, pi, ti, 0] = float(path[ti][0])
                    paths[gi, pi, ti, 1] = float(path[ti][1])

    return CometGroups(
        active=jnp.asarray(active),
        planet_ids=jnp.asarray(planet_ids),
        path_index=jnp.asarray(path_index),
        paths=jnp.asarray(paths),
        path_lengths=jnp.asarray(path_lengths),
    )


def pack_comet_planet_ids(ids: list[int]) -> tuple[jnp.ndarray, int]:
    arr = np.full((MAX_COMET_PLANETS,), -1, dtype=np.int32)
    n = min(len(ids), MAX_COMET_PLANETS)
    for i, pid in enumerate(ids[:n]):
        arr[i] = int(pid)
    return jnp.asarray(arr), n


def observation_to_state(
    obs: Any,
    *,
    episode_seed: int = 0,
    ship_speed: float = 6.0,
    episode_steps: int = 500,
    done: bool = False,
    rewards: tuple[float, float] = (0.0, 0.0),
    step_rewards: tuple[float, float] = (0.0, 0.0),
) -> OrbitWarsState:
    base = empty_state()
    planet_pad = np.zeros((MAX_PLANETS, PLANET_COLS), dtype=np.float32)
    fleet_pad = np.zeros((MAX_FLEETS, FLEET_COLS), dtype=np.float32)

    planets, n_planets = _pack_planets(list(_get(obs, "planets", []) or []), pad=planet_pad)
    initial, _ = _pack_planets(list(_get(obs, "initial_planets", []) or []), pad=planet_pad.copy())
    fleets, n_fleets = _pack_fleets(list(_get(obs, "fleets", []) or []), pad=fleet_pad)
    comets = pack_comets(list(_get(obs, "comets", []) or []))
    comet_ids, n_comet_ids = pack_comet_planet_ids(list(_get(obs, "comet_planet_ids", []) or []))

    return OrbitWarsState(
        planets=jnp.asarray(planets),
        initial_planets=jnp.asarray(initial),
        n_planets=jnp.int32(n_planets),
        fleets=jnp.asarray(fleets),
        n_fleets=jnp.int32(n_fleets),
        comets=comets,
        comet_planet_ids=comet_ids,
        n_comet_planet_ids=jnp.int32(n_comet_ids),
        angular_velocity=jnp.float32(float(_get(obs, "angular_velocity", 0.0))),
        step=jnp.int32(int(_get(obs, "step", 0))),
        next_fleet_id=jnp.int32(int(_get(obs, "next_fleet_id", 0))),
        episode_seed=jnp.int32(int(episode_seed)),
        done=jnp.bool_(done),
        rewards=jnp.asarray(rewards, dtype=jnp.float32),
        step_rewards=jnp.asarray(step_rewards, dtype=jnp.float32),
        ship_speed=jnp.float32(float(ship_speed)),
        episode_steps=jnp.int32(int(episode_steps)),
    )


def _planet_rows(state: OrbitWarsState) -> list[list[float]]:
    rows: list[list[float]] = []
    planets = np.asarray(state.planets)
    for i in range(int(state.n_planets)):
        if planets[i, 7] <= 0.0:
            continue
        rows.append(
            [
                float(planets[i, 0]),
                float(planets[i, 1]),
                float(planets[i, 2]),
                float(planets[i, 3]),
                float(planets[i, 4]),
                float(planets[i, 5]),
                float(planets[i, 6]),
            ]
        )
    return rows


def _fleet_rows(state: OrbitWarsState) -> list[list[float]]:
    rows: list[list[float]] = []
    fleets = np.asarray(state.fleets)
    for i in range(int(state.n_fleets)):
        if fleets[i, 7] <= 0.0:
            continue
        rows.append(
            [
                float(fleets[i, 0]),
                float(fleets[i, 1]),
                float(fleets[i, 2]),
                float(fleets[i, 3]),
                float(fleets[i, 4]),
                float(fleets[i, 5]),
                float(fleets[i, 6]),
            ]
        )
    return rows


def _unpack_comets(comets: CometGroups) -> list[dict[str, Any]]:
    active = np.asarray(comets.active)
    planet_ids = np.asarray(comets.planet_ids)
    path_index = np.asarray(comets.path_index)
    paths = np.asarray(comets.paths)
    path_lengths = np.asarray(comets.path_lengths)
    out: list[dict[str, Any]] = []
    for gi in range(MAX_COMET_GROUPS):
        if not active[gi]:
            continue
        group_paths: list[list[list[float]]] = []
        pids: list[int] = []
        for pi in range(4):
            pid = int(planet_ids[gi, pi])
            if pid < 0:
                continue
            pids.append(pid)
            plen = int(path_lengths[gi, pi])
            group_paths.append(
                [[float(paths[gi, pi, ti, 0]), float(paths[gi, pi, ti, 1])] for ti in range(plen)]
            )
        out.append({"planet_ids": pids, "paths": group_paths, "path_index": int(path_index[gi])})
    return out


def state_to_observation_dict(state: OrbitWarsState, *, player: int = 0) -> dict[str, Any]:
    comet_ids = [int(x) for x in np.asarray(state.comet_planet_ids)[: int(state.n_comet_planet_ids)] if int(x) >= 0]
    return {
        "step": int(state.step),
        "player": int(player),
        "planets": _planet_rows(state),
        "initial_planets": _planet_rows(
            OrbitWarsState(
                planets=state.initial_planets,
                initial_planets=state.initial_planets,
                n_planets=state.n_planets,
                fleets=state.fleets,
                n_fleets=state.n_fleets,
                comets=state.comets,
                comet_planet_ids=state.comet_planet_ids,
                n_comet_planet_ids=state.n_comet_planet_ids,
                angular_velocity=state.angular_velocity,
                step=state.step,
                next_fleet_id=state.next_fleet_id,
                episode_seed=state.episode_seed,
                done=state.done,
                rewards=state.rewards,
                step_rewards=state.step_rewards,
                ship_speed=state.ship_speed,
                episode_steps=state.episode_steps,
            )
        ),
        "fleets": _fleet_rows(state),
        "angular_velocity": float(state.angular_velocity),
        "next_fleet_id": int(state.next_fleet_id),
        "comets": _unpack_comets(state.comets),
        "comet_planet_ids": comet_ids,
    }


def states_equal(a: OrbitWarsState, b: OrbitWarsState, *, atol: float = 1e-4) -> bool:
    """Compare simulation-relevant fields (ignores padded inactive slots)."""
    if int(a.n_planets) != int(b.n_planets) or int(a.n_fleets) != int(b.n_fleets):
        return False
    if int(a.step) != int(b.step) or bool(a.done) != bool(b.done):
        return False
    ap = np.asarray(a.planets)[: int(a.n_planets), :7]
    bp = np.asarray(b.planets)[: int(b.n_planets), :7]
    af = np.asarray(a.fleets)[: int(a.n_fleets), :7]
    bf = np.asarray(b.fleets)[: int(b.n_fleets), :7]
    if not np.allclose(ap, bp, atol=atol, rtol=0.0):
        return False
    if not np.allclose(af, bf, atol=atol, rtol=0.0):
        return False
    return True


In [ ]:
%%writefile orbit_wars/decode.py
"""Pure-JAX geometry decoder for Orbit Wars actions.

Given a chosen `(source_planet, target_planet, bucket)` triple, produce a
legal `[from_id, angle, num_ships]` action row plus validity flags. All
operations are vectorized and vmap-friendly — Phase 4 (rollout) will broadcast
over batch × source.

Ship-bucket scheme (BUCKET_COUNT = 8):

    0  25%  of source ships    (min 4 ships)
    1  50%  of source ships
    2  75%  of source ships
    3 100%  of source ships    (all-in)
    4  target_ships + 1        (minimal capture)
    5  target_ships + 50% src  (capture with reserve)
    6  target_ships + inc_enemy - inc_allied + 1 (smart capture minimal)
    7  target_ships + inc_enemy - inc_allied + 25% src (smart capture reserve)

Buckets are masked invalid when the computed ship count is <= 0 or exceeds the
source planet's current ship count.

A move is masked invalid when:

- source planet is not active or not owned by `player`;
- target planet is not active;
- the chosen ship count is 0 or > source ships;
- the straight-line path from source to target crosses the sun;
- the source and target are the same (degenerate self-launch).
"""

from __future__ import annotations

import jax
import jax.numpy as jnp

from .constants import CENTER, SUN_RADIUS
from .geometry import (
    estimate_intercept_angles,
    is_orbiting_planet,
    point_to_segment_distance,
    safe_angle,
    sun_hit,
)
from .state import OrbitWarsState

BUCKET_COUNT = 8
MIN_LAUNCH_SHIPS = 4
SUN_PATH_MARGIN = 1.5
PATH_PLANET_MARGIN = 1.0
INTERCEPT_ITERATIONS = 5


# Launch offset to avoid spawning fleets inside the source planet.
LAUNCH_OFFSET_PADDING = 0.1


def ship_counts_for_buckets(
    source_ships: jnp.ndarray, target_ships: jnp.ndarray, incoming_me: jnp.ndarray, incoming_enemy: jnp.ndarray
) -> jnp.ndarray:
    """Return integer-valued ship counts for every bucket index.

    Inputs broadcast against each other. Output shape = broadcasted shape +
    `(BUCKET_COUNT,)`.
    """
    src = source_ships[..., None]
    tgt = target_ships[..., None]
    inc_me = incoming_me[..., None]
    inc_en = incoming_enemy[..., None]
    
    b0 = src * 0.25
    b1 = src * 0.50
    b2 = src * 0.75
    b3 = src * 1.00
    b4 = tgt + 1.0
    b5 = tgt + src * 0.50
    b6 = jnp.maximum(0.0, tgt + inc_en - inc_me) + 1.0
    b7 = jnp.maximum(0.0, tgt + inc_en - inc_me) + src * 0.25
    
    b0, b1, b2, b3, b4, b5, b6, b7 = jnp.broadcast_arrays(b0, b1, b2, b3, b4, b5, b6, b7)
    
    raw = jnp.concatenate([b0, b1, b2, b3, b4, b5, b6, b7], axis=-1)
    raw = jnp.maximum(raw, jnp.float32(MIN_LAUNCH_SHIPS))
    # Floor to int while keeping floats (the env stores ships as float32 ints).
    return jnp.floor(raw)


def bucket_validity_mask(
    ship_counts: jnp.ndarray, source_ships: jnp.ndarray
) -> jnp.ndarray:
    """Bool mask of buckets that can legally fire.

    `ship_counts` has the bucket dim last; `source_ships` is broadcast.
    """
    src = source_ships[..., None]
    return (ship_counts > 0.0) & (ship_counts <= src)


def path_crosses_sun(
    src_x: jnp.ndarray, src_y: jnp.ndarray,
    tgt_x: jnp.ndarray, tgt_y: jnp.ndarray,
    margin: float = 0.0,
) -> jnp.ndarray:
    """Does the straight segment src→tgt pass within SUN_RADIUS (+margin) of centre?"""
    return sun_hit(src_x, src_y, tgt_x, tgt_y, margin=margin)


def path_blocked_by_planets(
    start_x: jnp.ndarray,
    start_y: jnp.ndarray,
    target_x: jnp.ndarray,
    target_y: jnp.ndarray,
    planet_x: jnp.ndarray,
    planet_y: jnp.ndarray,
    planet_radius: jnp.ndarray,
    planet_active: jnp.ndarray,
    margin: float = PATH_PLANET_MARGIN,
) -> jnp.ndarray:
    """True when a third active planet intersects the start→target segment.

    Shape: inputs `(P_src, P_tgt)` for start/target; planet arrays `(P,)`.
    Returns `(P_src, P_tgt)`.
    """
    p = planet_x.shape[0]
    slot = jnp.arange(p)
    src_i = slot[:, None, None]
    tgt_i = slot[None, :, None]
    obs_i = slot[None, None, :]

    is_obstacle = (
        planet_active[None, None, :]
        & (obs_i != src_i)
        & (obs_i != tgt_i)
    )
    obs_r = (planet_radius + margin)[None, None, :]
    ox = planet_x[None, None, :]
    oy = planet_y[None, None, :]

    sx = start_x[:, :, None]
    sy = start_y[:, :, None]
    tx = target_x[:, :, None]
    ty = target_y[:, :, None]

    # Optimization: Bounding box check before expensive point-to-segment distance.
    # An obstacle can only block the path if it is within the segment's bounding box.
    x_min = jnp.minimum(sx, tx) - obs_r
    x_max = jnp.maximum(sx, tx) + obs_r
    y_min = jnp.minimum(sy, ty) - obs_r
    y_max = jnp.maximum(sy, ty) + obs_r
    
    in_box = (ox >= x_min) & (ox <= x_max) & (oy >= y_min) & (oy <= y_max)
    is_obstacle = is_obstacle & in_box

    d = point_to_segment_distance(ox, oy, sx, sy, tx, ty)
    return jnp.any((d <= obs_r) & is_obstacle, axis=2)


def launch_angle(
    src_x: jnp.ndarray, src_y: jnp.ndarray,
    tgt_x: jnp.ndarray, tgt_y: jnp.ndarray,
) -> jnp.ndarray:
    """`atan2(dy, dx)` aim. Intercept correction lives in a higher layer."""
    return jnp.arctan2(tgt_y - src_y, tgt_x - src_x)


def compose_action_grid(
    state: OrbitWarsState,
    player: jnp.int32 | int,
    *,
    intercept_iterations: int = INTERCEPT_ITERATIONS,
    enable_planet_block: bool = True,
    enable_incoming_projection: bool = True,
) -> dict[str, jnp.ndarray]:
    """Pre-compute everything the policy/rollout needs about every
    (source, target, bucket) triple in a single state.

    Returns a dict with all (P_src, P_tgt) / (P_src, P_tgt, BUCKETS) shaped
    arrays:

        source_valid   (P,)             bool          source planet owned by player
        target_valid   (P,)             bool          target planet is active
        angle          (P, P, BUCKETS)  float32       safe intercept aim per bucket
        sun_blocks     (P, P, BUCKETS)  bool          launch→aim crosses sun
        planet_blocks  (P, P, BUCKETS)  bool          another planet blocks path
        self_target    (P, P)           bool          true on the diagonal
        target_valid_pair (P, P)        bool          target_valid AND not self
        ship_counts    (P, P, BUCKETS)  float32       per-bucket ship count to send
        bucket_valid   (P, P, BUCKETS)  bool          ship count fits source's reserve
        pair_valid     (P, P)           bool          source_valid AND target_valid_pair
        full_valid     (P, P, BUCKETS)  bool          pair & bucket & !sun & !planet block
        from_ids       (P,)             float32       planet id per source slot
    """
    planets = state.planets
    active = planets[:, 7] > 0.0
    owner = planets[:, 1]
    player_f = jnp.float32(player)
    source_valid = active & (owner == player_f)
    target_valid = active

    x = planets[:, 2]
    y = planets[:, 3]
    radius = planets[:, 4]
    ships = planets[:, 5]

    tgt_orbiting = is_orbiting_planet(x, y, radius)  # (P,)

    if enable_incoming_projection:
        from .features_jax import _fleet_projections
        incoming_me, incoming_enemy, _, _ = _fleet_projections(state, player_f)
    else:
        incoming_me = jnp.zeros_like(ships)
        incoming_enemy = jnp.zeros_like(ships)

    src_ships_grid = ships[:, None]                  # (P, 1)
    tgt_ships_grid = ships[None, :]                  # (1, P)
    inc_me_grid = incoming_me[None, :]               # (1, P)
    inc_en_grid = incoming_enemy[None, :]            # (1, P)
    ship_counts = ship_counts_for_buckets(src_ships_grid, tgt_ships_grid, inc_me_grid, inc_en_grid)  # (P, P, B)

    p_count = planets.shape[0]
    bucket_axis = ship_counts.shape[-1]
    src_x_b = jnp.broadcast_to(x[:, None, None], (p_count, p_count, bucket_axis))
    src_y_b = jnp.broadcast_to(y[:, None, None], (p_count, p_count, bucket_axis))
    tgt_x_b = jnp.broadcast_to(x[None, :, None], (p_count, p_count, bucket_axis))
    tgt_y_b = jnp.broadcast_to(y[None, :, None], (p_count, p_count, bucket_axis))
    tgt_orb_b = jnp.broadcast_to(
        tgt_orbiting[None, :, None], (p_count, p_count, bucket_axis),
    )

    _raw_angle, aim_x, aim_y = estimate_intercept_angles(
        src_x_b, src_y_b, tgt_x_b, tgt_y_b, tgt_orb_b, ship_counts,
        state.angular_velocity, state.ship_speed, n_iter=intercept_iterations,
    )

    center_x = jnp.broadcast_to(x[:, None, None], (p_count, p_count, bucket_axis))
    center_y = jnp.broadcast_to(y[:, None, None], (p_count, p_count, bucket_axis))
    # Launch direction toward intercept; detour if the centre→aim segment crosses the sun.
    angle = safe_angle(center_x, center_y, aim_x, aim_y, sun_margin=SUN_PATH_MARGIN)

    src_radius_b = jnp.broadcast_to(radius[:, None, None], (p_count, p_count, bucket_axis))
    start_x = center_x + jnp.cos(angle) * (src_radius_b + LAUNCH_OFFSET_PADDING)
    start_y = center_y + jnp.sin(angle) * (src_radius_b + LAUNCH_OFFSET_PADDING)
    # Mask when centre→aim crosses the sun (matches heuristic pre-filter).
    sun_blocks = path_crosses_sun(center_x, center_y, aim_x, aim_y, margin=SUN_PATH_MARGIN)
    if enable_planet_block:
        center_x_2d = jnp.broadcast_to(x[:, None], (p_count, p_count))
        center_y_2d = jnp.broadcast_to(y[:, None], (p_count, p_count))
        tgt_x_2d = jnp.broadcast_to(x[None, :], (p_count, p_count))
        tgt_y_2d = jnp.broadcast_to(y[None, :], (p_count, p_count))
        pb_2d = path_blocked_by_planets(
            center_x_2d, center_y_2d, tgt_x_2d, tgt_y_2d, x, y, radius, active, margin=PATH_PLANET_MARGIN,
        )
        planet_blocks = jnp.broadcast_to(pb_2d[:, :, None], (p_count, p_count, bucket_axis))
    else:
        planet_blocks = jnp.zeros_like(sun_blocks, dtype=jnp.bool_)

    self_target = jnp.eye(planets.shape[0], dtype=jnp.bool_)
    target_valid_pair = target_valid[None, :]
    pair_valid = source_valid[:, None] & target_valid_pair

    bucket_valid = bucket_validity_mask(ship_counts, src_ships_grid)       # (P, P, B)
    full_valid = pair_valid[..., None] & bucket_valid & (~sun_blocks) & (~planet_blocks)

    from_ids = planets[:, 0]                         # (P,) float

    return {
        "source_valid": source_valid,
        "target_valid": target_valid,
        "angle": angle,
        "aim_x": aim_x,
        "aim_y": aim_y,
        "sun_blocks": sun_blocks,
        "planet_blocks": planet_blocks,
        "self_target": self_target,
        "target_valid_pair": target_valid_pair,
        "ship_counts": ship_counts,
        "bucket_valid": bucket_valid,
        "pair_valid": pair_valid,
        "full_valid": full_valid,
        "from_ids": from_ids,
    }


def pack_action_row(
    from_id: jnp.ndarray,
    angle: jnp.ndarray,
    ships: jnp.ndarray,
    valid: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Return `(row (3,), mask_scalar)` for one move.

    Invalid moves emit a zero row and mask=0.
    """
    row = jnp.stack([
        from_id.astype(jnp.float32),
        angle.astype(jnp.float32),
        jnp.floor(ships).astype(jnp.float32),
    ])
    valid_f = valid.astype(jnp.float32)
    return row * valid_f, valid_f


In [ ]:
%%writefile orbit_wars/env.py
"""High-level Orbit Wars JAX environment API."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from .convert import state_to_observation_dict
from .reference import reference_step
from .reset import reset
from .state import OrbitWarsState
from .step import step


@dataclass(slots=True)
class EnvStep:
    state: OrbitWarsState
    observation: dict[str, Any]
    rewards: tuple[float, float]
    done: bool


class OrbitWarsJaxEnv:
    """Single-environment wrapper matching RL training expectations."""

    def __init__(
        self,
        *,
        seed: int = 0,
        episode_steps: int = 500,
        ship_speed: float = 6.0,
        env_root: str | None = None,
        learner_player: int = 0,
    ) -> None:
        self.seed = int(seed)
        self.episode_steps = int(episode_steps)
        self.ship_speed = float(ship_speed)
        self.env_root = env_root
        self.learner_player = int(learner_player)
        self.state: OrbitWarsState | None = None
        self._episode = 0

    def reset(self, seed: int | None = None) -> dict[str, Any]:
        if seed is not None:
            self.seed = int(seed)
        else:
            self.seed = self.seed + 9973
        self.state = reset(
            self.seed,
            episode_steps=self.episode_steps,
            ship_speed=self.ship_speed,
            env_root=self.env_root,
        )
        self._episode += 1
        return state_to_observation_dict(self.state, player=self.learner_player)

    def step(self, learner_action: list[list[float | int]], opponent_action: list[list[float | int]]) -> EnvStep:
        if self.state is None:
            raise RuntimeError("Call reset() before step().")
        if self.learner_player == 0:
            actions = [learner_action, opponent_action]
        else:
            actions = [opponent_action, learner_action]
        self.state = step(self.state, actions)
        obs = state_to_observation_dict(self.state, player=self.learner_player)
        rewards = (float(self.state.rewards[0]), float(self.state.rewards[1]))
        done = bool(self.state.done)
        return EnvStep(state=self.state, observation=obs, rewards=rewards, done=done)


class VectorOrbitWarsEnv:
    """Batched env stepping for throughput benchmarks (JIT core, no comet spawn)."""

    def __init__(self, num_envs: int, *, episode_steps: int = 500) -> None:
        self.num_envs = int(num_envs)
        self.episode_steps = int(episode_steps)
        self.states: OrbitWarsState | None = None

    def reset_batch(self, seeds: list[int]) -> list[dict[str, Any]]:
        assert len(seeds) == self.num_envs
        import jax.numpy as jnp
        from .state import empty_state

        states = [reset(s, episode_steps=self.episode_steps) for s in seeds]
        # stack into batched struct — for benchmark use list stepping if stack fails
        self._state_list = states
        return [state_to_observation_dict(s, player=0) for s in states]

    def step_batch_noop(self) -> None:
        """Advance all envs with empty actions (benchmark helper)."""
        from .step import step

        empty: list[list[float | int]] = []
        self._state_list = [
            step(s, [empty, empty]) for s in self._state_list
        ]


In [ ]:
%%writefile orbit_wars/features_jax.py
"""Pure-JAX feature encoder for Orbit Wars.

This module converts an `OrbitWarsState` (or a batched `vmap` thereof) into
fixed-shape entity-level features:

- planet features:  (B, MAX_PLANETS, PLANET_FEATURE_DIM)  plus planet_mask  (B, MAX_PLANETS)
- fleet  features:  (B, MAX_FLEETS,  FLEET_FEATURE_DIM)   plus fleet_mask   (B, MAX_FLEETS)
- global features:  (B, GLOBAL_FEATURE_DIM)

All features are player-relative (perspective of `player`).

The encoder is JIT-friendly and vmappable. No NumPy bridge.
"""

from __future__ import annotations

import jax
import jax.numpy as jnp

from .constants import (
    BOARD_SIZE,
    CENTER,
    COMET_SPAWN_STEPS,
    MAX_COMET_GROUPS,
    MAX_FLEETS,
    MAX_PLANETS,
    ROTATION_RADIUS_LIMIT,
)
from .geometry import fleet_speed
from .state import OrbitWarsState

PLANET_FEATURE_DIM = 31
FLEET_FEATURE_DIM = 15
GLOBAL_FEATURE_DIM = 22

# Normalization constants.
SHIPS_LOG_DENOM = jnp.log1p(jnp.float32(5000.0))
FLEET_SHIPS_LOG_DENOM = jnp.log1p(jnp.float32(500.0))
PRODUCTION_DENOM = jnp.float32(5.0)
RADIUS_DENOM = jnp.float32(10.0)
DIST_DENOM = jnp.float32(50.0)


def _ships_log_norm(ships: jnp.ndarray) -> jnp.ndarray:
    """log1p(ships) / log1p(5000)."""
    return jnp.log1p(jnp.maximum(ships, 0.0)) / SHIPS_LOG_DENOM


def _fleet_ships_log_norm(ships: jnp.ndarray) -> jnp.ndarray:
    return jnp.log1p(jnp.maximum(ships, 0.0)) / FLEET_SHIPS_LOG_DENOM


def _is_comet_per_planet(state: OrbitWarsState) -> jnp.ndarray:
    """Boolean per-planet flag, vectorized."""
    pids = state.planets[:, 0].astype(jnp.int32)
    cpids = state.comet_planet_ids
    valid = cpids >= 0
    return jnp.any((pids[:, None] == cpids[None, :]) & valid[None, :], axis=-1)


def _is_orbiting_per_planet(state: OrbitWarsState) -> jnp.ndarray:
    init = state.initial_planets
    dx = init[:, 2] - CENTER
    dy = init[:, 3] - CENTER
    orbit_r = jnp.sqrt(dx * dx + dy * dy)
    radius = state.planets[:, 4]
    return orbit_r + radius < ROTATION_RADIUS_LIMIT


def _fleet_projections(state: OrbitWarsState, player_f: jnp.float32) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Calculate exact destination and ETA for all fleets using physics."""
    planets = state.planets
    fleets = state.fleets
    
    fx = fleets[:, 2]
    fy = fleets[:, 3]
    angle = fleets[:, 4]
    cos_a = jnp.cos(angle)
    sin_a = jnp.sin(angle)
    fleet_ships = fleets[:, 6]
    speed = fleet_speed(fleet_ships, state.ship_speed)
    fleet_active = fleets[:, 7] > 0.0

    px = planets[:, 2]
    py = planets[:, 3]
    radius = planets[:, 4]
    planet_active = planets[:, 7] > 0.0

    dx = px[None, :] - fx[:, None]     # (F, P)
    dy = py[None, :] - fy[:, None]     # (F, P)
    
    ux = cos_a[:, None]
    uy = sin_a[:, None]
    
    dot_ru = dx * ux + dy * uy
    
    sp = speed[:, None]
    sp_safe = jnp.maximum(sp, 1e-6)
    t_close_raw = dot_ru / sp_safe
    t_close = jnp.where(dot_ru > 0.0, t_close_raw, 0.0)
    
    cx = fx[:, None] + sp * ux * t_close
    cy = fy[:, None] + sp * uy * t_close
    
    d_close = jnp.sqrt((px[None, :] - cx)**2 + (py[None, :] - cy)**2)
    
    slop = jnp.float32(3.0)
    collides = (t_close > 0.0) & (d_close <= radius[None, :] + slop) & planet_active[None, :]
    
    big = jnp.float32(1e9)
    masked_t_close = jnp.where(collides, t_close, big)
    
    best_t_close = jnp.min(masked_t_close, axis=1) # (F,)
    best_p_idx = jnp.argmin(masked_t_close, axis=1) # (F,)
    
    has_target = fleet_active & (best_t_close < big)
    
    fleet_owner = fleets[:, 1]
    f_is_me = has_target & (fleet_owner == player_f)
    f_is_enemy = has_target & (fleet_owner >= 0.0) & (fleet_owner != player_f)
    
    incoming_me = jnp.zeros(MAX_PLANETS, dtype=jnp.float32)
    incoming_me = incoming_me.at[best_p_idx].add(jnp.where(f_is_me, fleet_ships, 0.0))
    
    incoming_enemy = jnp.zeros(MAX_PLANETS, dtype=jnp.float32)
    incoming_enemy = incoming_enemy.at[best_p_idx].add(jnp.where(f_is_enemy, fleet_ships, 0.0))
    
    eta_me = jnp.full(MAX_PLANETS, big, dtype=jnp.float32)
    eta_me = eta_me.at[best_p_idx].min(jnp.where(f_is_me, best_t_close, big))
    
    eta_enemy = jnp.full(MAX_PLANETS, big, dtype=jnp.float32)
    eta_enemy = eta_enemy.at[best_p_idx].min(jnp.where(f_is_enemy, best_t_close, big))
    
    eta_me_norm = jnp.minimum(eta_me, 100.0) / 100.0
    eta_enemy_norm = jnp.minimum(eta_enemy, 100.0) / 100.0
    
    return incoming_me, incoming_enemy, eta_me_norm, eta_enemy_norm


def _nearest_distance_to_subset(
    planets: jnp.ndarray,
    subset_mask: jnp.ndarray,
) -> jnp.ndarray:
    """For each planet, distance to the nearest other planet in `subset_mask`."""
    x = planets[:, 2]
    y = planets[:, 3]
    dx = x[:, None] - x[None, :]
    dy = y[:, None] - y[None, :]
    dist = jnp.sqrt(dx * dx + dy * dy)
    big = jnp.float32(1e6)
    self_mask = jnp.eye(MAX_PLANETS, dtype=jnp.bool_)
    masked = jnp.where(subset_mask[None, :] & (~self_mask), dist, big)
    nearest = jnp.min(masked, axis=-1)
    nearest = jnp.where(nearest >= big, DIST_DENOM * 3.0, nearest)
    return nearest


def _rank_norm(values: jnp.ndarray, mask: jnp.ndarray) -> jnp.ndarray:
    """Rank normalization in [0, 1]."""
    eligible_count = jnp.sum(mask.astype(jnp.float32))
    smaller = jnp.sum(
        (values[:, None] > values[None, :])
        & mask[None, :]
        & mask[:, None],
        axis=-1,
    ).astype(jnp.float32)
    denom = jnp.maximum(eligible_count - 1.0, 1.0)
    return jnp.where(mask, smaller / denom, 0.0)


def _comet_remaining_life(state: OrbitWarsState) -> jnp.ndarray:
    """Remaining comet path steps scattered to planet slots."""
    comets = state.comets
    cgpids = comets.planet_ids
    cplens = comets.path_lengths
    cactive = comets.active
    idx_next = comets.path_index + 1
    pids = state.planets[:, 0].astype(jnp.int32)
    active = state.planets[:, 7] > 0.0
    match_gqp = (cgpids[..., None] == pids[None, None, :]) & active[None, None, :] & (cgpids[..., None] >= 0)
    slot_gq = jnp.argmax(match_gqp.astype(jnp.int32), axis=-1)
    has_match_gq = jnp.any(match_gqp, axis=-1)
    remaining_gq = (cplens - idx_next[:, None]).astype(jnp.float32)
    valid_gq = has_match_gq & cactive[:, None] & (cgpids >= 0)
    out = jnp.zeros((MAX_PLANETS,), dtype=jnp.float32)
    flat_slot = slot_gq.reshape(-1)
    flat_val = jnp.where(valid_gq.reshape(-1), jnp.maximum(remaining_gq.reshape(-1), 0.0), 0.0)
    out = out.at[flat_slot].set(jnp.maximum(out[flat_slot], flat_val))
    return out


def encode_observation(
    state: OrbitWarsState,
    player: jnp.int32 | int,
) -> dict[str, jnp.ndarray]:
    """Full entity-level encoder."""
    player_f = jnp.float32(player)
    planets = state.planets
    fleets = state.fleets

    active = planets[:, 7] > 0.0
    owner = planets[:, 1]
    owner_is_me = active & (owner == player_f)
    owner_is_enemy = active & (owner >= 0.0) & (owner != player_f)
    owner_is_neutral = active & (owner < 0.0)

    ships = planets[:, 5]
    production = planets[:, 6]
    radius = planets[:, 4]
    x = planets[:, 2]
    y = planets[:, 3]

    dx_c = (x - CENTER) / DIST_DENOM
    dy_c = (y - CENTER) / DIST_DENOM
    dist_c = jnp.sqrt(dx_c * dx_c + dy_c * dy_c)

    is_orbiting = _is_orbiting_per_planet(state) & active
    is_comet = _is_comet_per_planet(state) & active

    init = state.initial_planets
    dx0 = init[:, 2] - CENTER
    dy0 = init[:, 3] - CENTER
    orbit_r = jnp.sqrt(dx0 * dx0 + dy0 * dy0)
    initial_angle = jnp.arctan2(dy0, dx0)
    current_angle = initial_angle + state.angular_velocity * state.step.astype(jnp.float32)
    orbit_angle_sin = jnp.where(is_orbiting, jnp.sin(current_angle), 0.0)
    orbit_angle_cos = jnp.where(is_orbiting, jnp.cos(current_angle), 0.0)
    orbit_r_norm = jnp.where(is_orbiting, orbit_r / DIST_DENOM, 0.0)

    incoming_me, incoming_enemy, eta_me_norm, eta_enemy_norm = _fleet_projections(state, player_f)

    nearest_enemy_d_raw = _nearest_distance_to_subset(planets, owner_is_enemy)
    nearest_friend_d_raw = _nearest_distance_to_subset(planets, owner_is_me)
    nearest_neutral_d_raw = _nearest_distance_to_subset(planets, owner_is_neutral)
    nearest_enemy_d = nearest_enemy_d_raw / DIST_DENOM
    nearest_friend_d = nearest_friend_d_raw / DIST_DENOM

    max_speed = state.ship_speed
    time_norm = jnp.float32(100.0)
    time_to_nearest_enemy = jnp.where(active, nearest_enemy_d_raw / max_speed / time_norm, 0.0)
    time_to_nearest_neutral = jnp.where(active, nearest_neutral_d_raw / max_speed / time_norm, 0.0)

    roi = production / (ships + 1.0)
    roi_norm = roi / jnp.float32(2.0)
    is_high_value = (production >= 3.0).astype(jnp.float32) * active.astype(jnp.float32)

    ship_rank_all = _rank_norm(ships, active)
    prod_rank_all = _rank_norm(production, active)
    my_ship_rank = _rank_norm(ships, owner_is_me)
    enemy_ship_rank = _rank_norm(ships, owner_is_enemy)
    is_my_largest = owner_is_me & (my_ship_rank >= jnp.float32(1.0 - 1e-6))
    is_enemy_largest = owner_is_enemy & (enemy_ship_rank >= jnp.float32(1.0 - 1e-6))

    net_balance = ships + incoming_me - incoming_enemy
    net_balance_signed_log = jnp.sign(net_balance) * _ships_log_norm(jnp.abs(net_balance))
    would_lose = active & owner_is_me & (incoming_enemy > ships)

    comet_remaining = _comet_remaining_life(state)
    comet_remaining_norm = comet_remaining / jnp.float32(64.0)

    # ---------------- Global features --------------------------------------
    my_planet_count = jnp.sum(owner_is_me.astype(jnp.float32))
    enemy_planet_count = jnp.sum(owner_is_enemy.astype(jnp.float32))
    neutral_planet_count = jnp.sum(owner_is_neutral.astype(jnp.float32))
    my_prod = jnp.sum(jnp.where(owner_is_me, production, 0.0))
    enemy_prod = jnp.sum(jnp.where(owner_is_enemy, production, 0.0))
    my_ships_total = jnp.sum(jnp.where(owner_is_me, ships, 0.0))
    enemy_ships_total = jnp.sum(jnp.where(owner_is_enemy, ships, 0.0))
    prod_lead = (my_prod - enemy_prod) / (my_prod + enemy_prod + 1.0)
    ship_lead = (my_ships_total - enemy_ships_total) / (my_ships_total + enemy_ships_total + 1.0)
    active_comets = jnp.sum(state.comets.active.astype(jnp.float32))
    turn = state.step.astype(jnp.float32) / jnp.maximum(state.episode_steps.astype(jnp.float32), 1.0)
    
    cur_step = state.step.astype(jnp.float32)
    ep = jnp.maximum(state.episode_steps.astype(jnp.float32), 1.0)
    spawn_arr = jnp.asarray(COMET_SPAWN_STEPS, dtype=jnp.float32)
    delta = spawn_arr - cur_step
    delta = jnp.where(delta > 0.0, delta, ep)
    next_comet_in = jnp.min(delta) / ep

    global_features = jnp.stack([
        turn, 1.0 - turn,
        my_planet_count / MAX_PLANETS, enemy_planet_count / MAX_PLANETS, neutral_planet_count / MAX_PLANETS,
        my_prod / 50.0, enemy_prod / 50.0,
        prod_lead, ship_lead,
        active_comets / 3.0, next_comet_in
    ], axis=-1)
    # Pad to GLOBAL_FEATURE_DIM if needed, here it's 11.
    global_features = jnp.pad(global_features, (0, GLOBAL_FEATURE_DIM - global_features.shape[0]))

    planet_features = jnp.stack([
        active.astype(jnp.float32), owner_is_me.astype(jnp.float32), owner_is_enemy.astype(jnp.float32), owner_is_neutral.astype(jnp.float32),
        _ships_log_norm(ships), production / PRODUCTION_DENOM, radius / RADIUS_DENOM, x / BOARD_SIZE, y / BOARD_SIZE,
        dx_c, dy_c, dist_c, is_orbiting.astype(jnp.float32), is_comet.astype(jnp.float32), orbit_r_norm, orbit_angle_sin, orbit_angle_cos,
        _ships_log_norm(incoming_me), _ships_log_norm(incoming_enemy), eta_me_norm, eta_enemy_norm, roi_norm, is_high_value,
        nearest_enemy_d - nearest_friend_d, ship_rank_all, prod_rank_all, net_balance_signed_log, would_lose.astype(jnp.float32),
        time_to_nearest_enemy, time_to_nearest_neutral, comet_remaining_norm
    ], axis=-1)
    planet_features = jnp.where(active[:, None], planet_features, 0.0)

    # ---------------- Fleet features --------------------------------------
    fleet_owner = fleets[:, 1]
    fleet_ships = fleets[:, 6]
    fleet_active = fleets[:, 7] > 0.0
    f_is_me = fleet_active & (fleet_owner == player_f)
    f_is_enemy = fleet_active & (fleet_owner >= 0.0) & (fleet_owner != player_f)
    fspeed = fleet_speed(fleet_ships, state.ship_speed)
    fx = fleets[:, 2]
    fy = fleets[:, 3]
    fangle = fleets[:, 4]
    
    fleet_features = jnp.stack([
        fleet_active.astype(jnp.float32), f_is_me.astype(jnp.float32), f_is_enemy.astype(jnp.float32),
        _fleet_ships_log_norm(fleet_ships), jnp.cos(fangle), jnp.sin(fangle), fx / BOARD_SIZE, fy / BOARD_SIZE,
        fspeed / state.ship_speed, (fx - CENTER) / DIST_DENOM, (fy - CENTER) / DIST_DENOM
    ], axis=-1)
    fleet_features = jnp.pad(fleet_features, ((0, 0), (0, FLEET_FEATURE_DIM - fleet_features.shape[-1])))
    fleet_features = jnp.where(fleet_active[:, None], fleet_features, 0.0)

    return {
        "planet_features": planet_features,
        "planet_mask": active,
        "fleet_features": fleet_features,
        "fleet_mask": fleet_active,
        "global_features": global_features,
    }

def encode_batch(states: OrbitWarsState, players: jnp.ndarray) -> dict[str, jnp.ndarray]:
    return jax.vmap(encode_observation, in_axes=(0, 0))(states, players)

encode_batch_jit = jax.jit(encode_batch)
encode_observation_jit = jax.jit(encode_observation)


In [ ]:
%%writefile orbit_wars/geometry.py
"""Pure JAX geometry helpers for Orbit Wars fleet/planet collision."""

from __future__ import annotations

import jax
import jax.numpy as jnp

from .constants import BOARD_SIZE, CENTER, ROTATION_RADIUS_LIMIT, SUN_RADIUS


def distance_xy(x1: jnp.ndarray, y1: jnp.ndarray, x2: jnp.ndarray, y2: jnp.ndarray) -> jnp.ndarray:
    return jnp.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)


def point_to_segment_distance(
    px: jnp.ndarray,
    py: jnp.ndarray,
    ax: jnp.ndarray,
    ay: jnp.ndarray,
    bx: jnp.ndarray,
    by: jnp.ndarray,
) -> jnp.ndarray:
    dx = bx - ax
    dy = by - ay
    denom = dx * dx + dy * dy
    t = jnp.where(
        denom > 0.0,
        ((px - ax) * dx + (py - ay) * dy) / denom,
        0.0,
    )
    t = jnp.clip(t, 0.0, 1.0)
    cx = ax + t * dx
    cy = ay + t * dy
    return jnp.sqrt((px - cx) ** 2 + (py - cy) ** 2)


def swept_pair_hit(
    ax: jnp.ndarray,
    ay: jnp.ndarray,
    bx: jnp.ndarray,
    by: jnp.ndarray,
    p0x: jnp.ndarray,
    p0y: jnp.ndarray,
    p1x: jnp.ndarray,
    p1y: jnp.ndarray,
    radius: jnp.ndarray,
) -> jnp.ndarray:
    d0x = ax - p0x
    d0y = ay - p0y
    dvx = (bx - ax) - (p1x - p0x)
    dvy = (by - ay) - (p1y - p0y)
    a = dvx * dvx + dvy * dvy
    b = 2.0 * (d0x * dvx + d0y * dvy)
    c = d0x * d0x + d0y * d0y - radius * radius
    disc = b * b - 4.0 * a * c
    no_motion = a < 1e-12
    hit_no_motion = c <= 0.0
    sq = jnp.sqrt(jnp.maximum(disc, 0.0))
    t1 = (-b - sq) / (2.0 * a + 1e-12)
    t2 = (-b + sq) / (2.0 * a + 1e-12)
    hit_motion = (disc >= 0.0) & (t2 >= 0.0) & (t1 <= 1.0)
    return jnp.where(no_motion, hit_no_motion, hit_motion)


def fleet_speed(ships: jnp.ndarray, max_speed: jnp.ndarray) -> jnp.ndarray:
    log1000 = jnp.log(1000.0)
    speed = 1.0 + (max_speed - 1.0) * (jnp.log(jnp.maximum(ships, 1.0)) / log1000) ** 1.5
    return jnp.minimum(speed, max_speed)


def sun_hit(old_x, old_y, new_x, new_y, margin: float = 0.0) -> jnp.ndarray:
    return point_to_segment_distance(
        jnp.float32(CENTER), jnp.float32(CENTER), old_x, old_y, new_x, new_y,
    ) < (SUN_RADIUS + margin)


def segment_intersects_circle(
    ax: jnp.ndarray,
    ay: jnp.ndarray,
    bx: jnp.ndarray,
    by: jnp.ndarray,
    cx: jnp.ndarray,
    cy: jnp.ndarray,
    radius: jnp.ndarray,
) -> jnp.ndarray:
    d = point_to_segment_distance(cx, cy, ax, ay, bx, by)
    return d <= radius


def segment_clear_of_circles(
    ax: jnp.ndarray,
    ay: jnp.ndarray,
    bx: jnp.ndarray,
    by: jnp.ndarray,
    cx: jnp.ndarray,
    cy: jnp.ndarray,
    radius: jnp.ndarray,
    valid: jnp.ndarray,
) -> jnp.ndarray:
    """True when the segment does not intersect any valid circle."""
    blocked = segment_intersects_circle(ax, ay, bx, by, cx, cy, radius)
    return ~jnp.any(blocked & valid, axis=-1)


def _angle_diff(a: jnp.ndarray, b: jnp.ndarray) -> jnp.ndarray:
    dd = (a - b) % (2.0 * jnp.pi)
    return jnp.minimum(dd, 2.0 * jnp.pi - dd)


def safe_angle(
    src_x: jnp.ndarray,
    src_y: jnp.ndarray,
    aim_x: jnp.ndarray,
    aim_y: jnp.ndarray,
    sun_margin: float = 1.5,
) -> jnp.ndarray:
    """Return a launch angle from source to aim, detouring around the sun if needed."""
    src_x = src_x.astype(jnp.float64)
    src_y = src_y.astype(jnp.float64)
    aim_x = aim_x.astype(jnp.float64)
    aim_y = aim_y.astype(jnp.float64)
    margin = jnp.float64(sun_margin)
    sun_r = jnp.float64(SUN_RADIUS)
    center = jnp.float64(CENTER)

    direct = jnp.arctan2(aim_y - src_y, aim_x - src_x)
    crosses = sun_hit(src_x, src_y, aim_x, aim_y, margin=float(sun_margin))
    d = jnp.sqrt((src_x - center) ** 2 + (src_y - center) ** 2)
    inside = d <= sun_r + 1.0
    half = jnp.arcsin(jnp.minimum(1.0, (sun_r + margin) / jnp.maximum(d, 1e-6)))
    to_sun = jnp.arctan2(center - src_y, center - src_x)
    cw = to_sun + half
    ccw = to_sun - half
    detour = jnp.where(_angle_diff(cw, direct) <= _angle_diff(ccw, direct), cw, ccw)
    return jnp.where(crosses & ~inside, detour, direct).astype(jnp.float32)


def predict_orbit_polar(
    x: jnp.ndarray,
    y: jnp.ndarray,
    angular_velocity: jnp.ndarray,
    turns_ahead: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Match heuristic notebook: advance polar angle around the sun."""
    theta = jnp.arctan2(y - CENTER, x - CENTER)
    r = jnp.sqrt((x - CENTER) ** 2 + (y - CENTER) ** 2)
    theta2 = theta + angular_velocity * turns_ahead
    return CENTER + r * jnp.cos(theta2), CENTER + r * jnp.sin(theta2)


def in_bounds(x: jnp.ndarray, y: jnp.ndarray) -> jnp.ndarray:
    return (x >= 0.0) & (x <= BOARD_SIZE) & (y >= 0.0) & (y <= BOARD_SIZE)


def rotate_around_center(x: jnp.ndarray, y: jnp.ndarray, radians: jnp.ndarray) -> tuple[jnp.ndarray, jnp.ndarray]:
    dx = x - CENTER
    dy = y - CENTER
    c = jnp.cos(radians)
    s = jnp.sin(radians)
    return CENTER + dx * c - dy * s, CENTER + dx * s + dy * c


def is_orbiting_planet(x: jnp.ndarray, y: jnp.ndarray, radius: jnp.ndarray) -> jnp.ndarray:
    orbit_r = jnp.sqrt((x - CENTER) ** 2 + (y - CENTER) ** 2)
    return orbit_r + radius < ROTATION_RADIUS_LIMIT


def predict_planet_position(
    x: jnp.ndarray,
    y: jnp.ndarray,
    is_orbiting: jnp.ndarray,
    turns_ahead: jnp.ndarray,
    angular_velocity: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    px, py = predict_orbit_polar(x, y, angular_velocity, turns_ahead)
    return jnp.where(is_orbiting, px, x), jnp.where(is_orbiting, py, y)


def solve_intercept(
    src_x: jnp.ndarray,
    src_y: jnp.ndarray,
    tgt_x: jnp.ndarray,
    tgt_y: jnp.ndarray,
    tgt_is_orbiting: jnp.ndarray,
    ship_count: jnp.ndarray,
    angular_velocity: jnp.ndarray,
    max_speed: jnp.ndarray,
    n_iter: int = 25,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Return `(aim_x, aim_y, travel_time)` for a fleet of `ship_count` ships."""
    speed = fleet_speed(ship_count, max_speed)
    dist0 = distance_xy(src_x, src_y, tgt_x, tgt_y)
    travel_time = dist0 / jnp.maximum(speed, 1e-6)

    def body(_i, carry):
        tt, _ix, _iy = carry
        ix, iy = predict_planet_position(tgt_x, tgt_y, tgt_is_orbiting, tt, angular_velocity)
        tt_new = distance_xy(src_x, src_y, ix, iy) / jnp.maximum(speed, 1e-6)
        return tt_new, ix, iy

    ix0, iy0 = predict_planet_position(tgt_x, tgt_y, tgt_is_orbiting, travel_time, angular_velocity)
    travel_time, aim_x, aim_y = jax.lax.fori_loop(0, n_iter, body, (travel_time, ix0, iy0))
    return aim_x, aim_y, travel_time


def estimate_intercept_angles(
    src_x: jnp.ndarray,
    src_y: jnp.ndarray,
    tgt_x: jnp.ndarray,
    tgt_y: jnp.ndarray,
    tgt_is_orbiting: jnp.ndarray,
    ship_counts: jnp.ndarray,
    angular_velocity: jnp.ndarray,
    max_speed: jnp.ndarray,
    n_iter: int = 25,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Iterative lead-angle estimate matching the heuristic bot.

    All inputs broadcast to a common shape ending with optional bucket dim.
    Returns `(angle, aim_x, aim_y)` with the same broadcast shape.
    """
    speed = fleet_speed(ship_counts, max_speed)
    dist0 = distance_xy(src_x, src_y, tgt_x, tgt_y)
    travel_time = dist0 / jnp.maximum(speed, 1e-6)
    pred_x, pred_y = predict_planet_position(
        tgt_x, tgt_y, tgt_is_orbiting, travel_time, angular_velocity,
    )

    def body(_i, carry):
        px, py, _tt = carry
        dist = distance_xy(src_x, src_y, px, py)
        tt_new = dist / jnp.maximum(speed, 1e-6)
        nx, ny = predict_planet_position(tgt_x, tgt_y, tgt_is_orbiting, tt_new, angular_velocity)
        return nx, ny, tt_new

    pred_x, pred_y, _ = jax.lax.fori_loop(0, n_iter, body, (pred_x, pred_y, travel_time))
    angle = jnp.arctan2(pred_y - src_y, pred_x - src_x)
    return angle, pred_x, pred_y


In [ ]:
%%writefile orbit_wars/heuristic_opponent.py
"""Load the frozen kaggle700 heuristic and produce padded action tensors."""

from __future__ import annotations

import importlib.util
import sys
from pathlib import Path
from typing import Any, Callable

import jax.numpy as jnp
import numpy as np

from .constants import MAX_MOVES_PER_PLAYER
from .convert import state_to_observation_dict
from .state import OrbitWarsState


def default_heuristic_path() -> Path:
    """Resolve `versions/kaggle700_current_heuristic/main.py` from repo root."""
    here = Path(__file__).resolve()
    for root in here.parents:
        candidate = root / "versions" / "kaggle700_current_heuristic" / "main.py"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find versions/kaggle700_current_heuristic/main.py. "
        "Include `versions/` in your Kaggle dataset or git clone."
    )


def _resolve_custom_path(path: Path) -> Path:
    """Resolve a user-supplied path relative to cwd or repo root."""
    if path.is_absolute():
        return path
    if path.exists():
        return path.resolve()
    here = Path(__file__).resolve()
    for root in here.parents:
        candidate = root / path
        if candidate.exists():
            return candidate
    return path


def load_heuristic_agent(path: Path | None = None) -> Callable[[Any], list]:
    """Import and return the heuristic `agent(obs)` function."""
    if path is None:
        bot_path = default_heuristic_path().resolve()
    else:
        bot_path = _resolve_custom_path(Path(path)).resolve()
    heur_root = bot_path.parent
    if str(heur_root) not in sys.path:
        sys.path.insert(0, str(heur_root))
    spec = importlib.util.spec_from_file_location("heuristic_main", bot_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Failed to load heuristic from {bot_path}")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.agent


def pack_moves_list(moves: list[list[float | int]]) -> tuple[np.ndarray, np.ndarray]:
    """Pack Kaggle-style moves into `(MAX_MOVES, 3)` + mask."""
    actions = np.zeros((MAX_MOVES_PER_PLAYER, 3), dtype=np.float32)
    mask = np.zeros((MAX_MOVES_PER_PLAYER,), dtype=np.float32)
    for i, row in enumerate(moves[:MAX_MOVES_PER_PLAYER]):
        actions[i, 0] = float(row[0])
        actions[i, 1] = float(row[1])
        actions[i, 2] = float(row[2])
        mask[i] = 1.0
    return actions, mask


def heuristic_actions_for_state(state: OrbitWarsState, player: int, agent) -> tuple[np.ndarray, np.ndarray]:
    obs = state_to_observation_dict(state, player=int(player))
    moves = agent(obs)
    return pack_moves_list(moves)


def batched_heuristic_actions(
    states: OrbitWarsState,
    opponent_players: np.ndarray,
    agent,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Build padded `(B, M, 3)` action tensors for player 0 and player 1.

    `opponent_players[i]` is the heuristic seat for env *i* (0 or 1).
    Non-heuristic seats are zero with mask 0.
    """
    import jax.tree_util as tu

    n = int(states.step.shape[0])
    a0 = np.zeros((n, MAX_MOVES_PER_PLAYER, 3), dtype=np.float32)
    m0 = np.zeros((n, MAX_MOVES_PER_PLAYER), dtype=np.float32)
    a1 = np.zeros((n, MAX_MOVES_PER_PLAYER, 3), dtype=np.float32)
    m1 = np.zeros((n, MAX_MOVES_PER_PLAYER), dtype=np.float32)

    for i in range(n):
        single = tu.tree_map(lambda x, i=i: x[i], states)
        opp = int(opponent_players[i])
        act, msk = heuristic_actions_for_state(single, opp, agent)
        if opp == 0:
            a0[i] = act
            m0[i] = msk
        else:
            a1[i] = act
            m1[i] = msk

    return jnp.asarray(a0), jnp.asarray(m0), jnp.asarray(a1), jnp.asarray(m1)


In [ ]:
%%writefile orbit_wars/reference.py
"""Reference Kaggle env bridge for reset and parity validation."""

from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path
from types import ModuleType
from typing import Any


def load_orbit_wars_module() -> ModuleType:
    """Import official orbit_wars.py (installed package or local checkout)."""
    try:
        from kaggle_environments.envs.orbit_wars import orbit_wars as ref

        return ref
    except ImportError:
        pass

    candidates = [
        Path("/media/yahor/ADATA SE880/datasets/kaggle-environments-master"),
        Path(__file__).resolve().parents[3] / "analysis" / "fast_kaggle_env",
    ]
    for root in candidates:
        module_path = root / "kaggle_environments" / "envs" / "orbit_wars" / "orbit_wars.py"
        if module_path.exists():
            if str(root) not in sys.path:
                sys.path.insert(0, str(root))
            from kaggle_environments.envs.orbit_wars import orbit_wars as ref

            return ref

    raise ImportError(
        "Could not import kaggle_environments.envs.orbit_wars.orbit_wars. "
        "On Kaggle this should be preinstalled; locally install kaggle-environments "
        "or set analysis/fast_kaggle_env."
    )


@dataclass(slots=True)
class ReferenceStep:
    observations: list[Any]
    rewards: tuple[float, float]
    done: bool


def add_env_root(env_root: str | Path) -> Path:
    path = Path(env_root)
    if not path.is_absolute():
        path = (Path.cwd() / path).resolve()
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))
    return path


def default_env_root() -> Path:
    repo = Path(__file__).resolve().parents[2]
    fast = repo / "analysis" / "fast_kaggle_env"
    official = Path("/media/yahor/ADATA SE880/datasets/kaggle-environments-master")
    if fast.exists():
        return fast
    if official.exists():
        return official
    return fast


def make_reference_env(
    *,
    seed: int,
    episode_steps: int = 500,
    env_root: str | Path | None = None,
) -> Any:
    root = add_env_root(env_root or default_env_root())
    from kaggle_environments import make

    configuration = {"episodeSteps": int(episode_steps), "seed": int(seed), "randomSeed": int(seed)}
    env = make("orbit_wars", configuration=configuration, debug=False)
    env.reset(num_agents=2)
    return env


def extract_observation(state: Any) -> Any:
    if isinstance(state, dict):
        return state.get("observation")
    return getattr(state, "observation")


def extract_reward(state: Any) -> float:
    if isinstance(state, dict):
        value = state.get("reward", 0.0)
    else:
        value = getattr(state, "reward", 0.0)
    return 0.0 if value is None else float(value)


def extract_status(state: Any) -> str:
    if isinstance(state, dict):
        return str(state.get("status", "UNKNOWN"))
    return str(getattr(state, "status", "UNKNOWN"))


def reference_reset(seed: int, *, episode_steps: int = 500, env_root: str | Path | None = None) -> tuple[Any, ReferenceStep]:
    env = make_reference_env(seed=seed, episode_steps=episode_steps, env_root=env_root)
    states = env.step([[], []])
    obs = [extract_observation(states[i]) for i in range(2)]
    rewards = (extract_reward(states[0]), extract_reward(states[1]))
    done = extract_status(states[0]) != "ACTIVE"
    return env, ReferenceStep(observations=obs, rewards=rewards, done=done)


def reference_step(env: Any, actions: list[list[list[float | int]]]) -> ReferenceStep:
    states = env.step(actions)
    obs = [extract_observation(states[i]) for i in range(2)]
    rewards = (extract_reward(states[0]), extract_reward(states[1]))
    done = extract_status(states[0]) != "ACTIVE"
    return ReferenceStep(observations=obs, rewards=rewards, done=done)


def episode_seed_from_env(env: Any) -> int:
    info = getattr(env, "info", None) or {}
    seed = info.get("seed")
    if seed is not None:
        return int(seed)
    return 0


In [ ]:
%%writefile orbit_wars/reset.py
"""Reset Orbit Wars JAX state from reference env."""

from __future__ import annotations

from .convert import observation_to_state
from .reference import episode_seed_from_env, reference_reset
from .state import OrbitWarsState


def reset(
    seed: int,
    *,
    episode_steps: int = 500,
    ship_speed: float = 6.0,
    env_root: str | None = None,
) -> OrbitWarsState:
    env, ref = reference_reset(seed, episode_steps=episode_steps, env_root=env_root)
    episode_seed = episode_seed_from_env(env)
    return observation_to_state(
        ref.observations[0],
        episode_seed=episode_seed,
        ship_speed=ship_speed,
        episode_steps=episode_steps,
        done=ref.done,
        rewards=ref.rewards,
    )


In [ ]:
%%writefile orbit_wars/rollout.py
"""Sample masked actions from the Transformer policy and pack them for `step_jit`.

Wiring:

    state, params -> features (Phase 1)
                  -> policy out (Phase 2)
                  -> decode grid (Phase 3)
                  -> apply masks, sample target & bucket  (HERE)
                  -> pack (M, 3) action tensor + mask     (HERE)

Sampling is two-stage:

1. For every owned source planet, sample a target slot from the masked
   target logits (categorical with -inf on invalid targets).
2. Conditional on the chosen target, sample a bucket from masked bucket
   logits.

If a source planet has *no* valid (target, bucket) combination, its move is
masked out at the action-packing step.

We also compute the joint log-probability and per-row entropy needed by PPO.
"""

from __future__ import annotations

import jax
import jax.numpy as jnp

from .constants import MAX_MOVES_PER_PLAYER, MAX_PLANETS
from .decode import BUCKET_COUNT, compose_action_grid

_NEG_INF = jnp.float32(-1e9)


def _masked_log_softmax(logits: jnp.ndarray, mask: jnp.ndarray) -> jnp.ndarray:
    """log_softmax with -inf for masked entries.

    If ALL entries are masked, we return a safe uniform log-prob so downstream
    sampling doesn't NaN; the caller still knows the source is invalid via the
    source-level mask.
    """
    any_valid = jnp.any(mask, axis=-1, keepdims=True)
    safe_logits = jnp.where(mask, logits, _NEG_INF)
    # Replace fully-masked rows with zero logits to avoid -inf log_softmax NaN.
    safe_logits = jnp.where(any_valid, safe_logits, jnp.zeros_like(logits))
    return jax.nn.log_softmax(safe_logits, axis=-1)


def _entropy_from_log_probs(log_probs: jnp.ndarray, mask: jnp.ndarray) -> jnp.ndarray:
    """Sum -p log p over masked entries (axis=-1)."""
    p = jnp.exp(log_probs) * mask.astype(log_probs.dtype)
    return -jnp.sum(p * log_probs, axis=-1)


def sample_actions(
    rng: jax.Array,
    target_logits: jnp.ndarray,    # (B, P, P)
    bucket_logits: jnp.ndarray,    # (B, P, BUCKETS)
    action_grid: dict,             # output of compose_action_grid per env (vmapped: leading B)
    deterministic: bool = False,
) -> dict[str, jnp.ndarray]:
    """Sample (target, bucket) per source planet with full masking.

    Inputs are batched (`B` = num envs). `action_grid` should be the vmapped
    output of `compose_action_grid` (i.e. each value has a leading `B` axis).

    Returns dict with:
        target_idx       (B, P) int32         chosen target slot per source
        bucket_idx       (B, P) int32         chosen bucket per source
        log_prob         (B, P) float32       joint log-prob of (target, bucket)
        entropy_target   (B, P) float32       entropy of target distribution
        entropy_bucket   (B, P) float32       entropy of bucket dist (under chosen target)
        source_valid     (B, P) bool          source planet is owned AND has >=1 valid action
        target_valid_any (B, P) bool          at least one valid target for this source
    """
    pair_valid = action_grid["pair_valid"]          # (B, P, P)
    full_valid = action_grid["full_valid"]           # (B, P, P, BUCKETS)
    source_owned = action_grid["source_valid"]       # (B, P)

    # A target is choosable if any bucket is legal for that (source, target).
    target_has_bucket = jnp.any(full_valid, axis=-1)     # (B, P, P)
    target_valid_any = jnp.any(target_has_bucket, axis=-1)              # (B, P)
    source_valid = source_owned & target_valid_any                      # (B, P)

    # Target distribution: log_softmax with mask.
    tgt_log_probs = _masked_log_softmax(target_logits, target_has_bucket)
    entropy_target = _entropy_from_log_probs(tgt_log_probs, target_has_bucket)

    # Sample target. We split rng across (B, P) by folding indices.
    b, p, _ = target_logits.shape
    rng, k_tgt = jax.random.split(rng)
    tgt_keys = jax.random.split(k_tgt, b * p).reshape(b, p, 2)

    def _sample_target(lp_row, key):
        if deterministic:
            return jnp.argmax(lp_row, axis=-1)
        return jax.random.categorical(key, lp_row, axis=-1)

    target_idx = jax.vmap(jax.vmap(_sample_target))(tgt_log_probs, tgt_keys).astype(jnp.int32)
    # Force a safe value when source has no valid action (we'll mask the row
    # out at packing time anyway).
    target_idx = jnp.where(source_valid, target_idx, jnp.int32(0))

    tgt_lp = jnp.take_along_axis(tgt_log_probs, target_idx[..., None], axis=-1).squeeze(-1)
    tgt_lp = jnp.where(source_valid, tgt_lp, jnp.float32(0.0))

    # Bucket distribution conditional on the chosen target.
    # Index into the (B, P, P, BUCKETS) bucket_logits using target_idx.
    b_idx = jnp.arange(b)[:, None]
    p_idx = jnp.arange(p)[None, :]
    chosen_bucket_logits = bucket_logits[b_idx, p_idx, target_idx] # (B, P, BUCKETS)

    # Gather full_valid[b, s, target_idx[b, s], :] -> (B, P, BUCKETS)
    chosen_bucket_valid = jnp.take_along_axis(
        full_valid, target_idx[..., None, None].repeat(BUCKET_COUNT, axis=-1), axis=2
    ).squeeze(2)                                                # (B, P, BUCKETS)
    bucket_lp_row = _masked_log_softmax(chosen_bucket_logits, chosen_bucket_valid)
    entropy_bucket = _entropy_from_log_probs(bucket_lp_row, chosen_bucket_valid)

    rng, k_bkt = jax.random.split(rng)
    bkt_keys = jax.random.split(k_bkt, b * p).reshape(b, p, 2)

    def _sample_bucket(lp_row, key):
        if deterministic:
            return jnp.argmax(lp_row, axis=-1)
        return jax.random.categorical(key, lp_row, axis=-1)

    bucket_idx = jax.vmap(jax.vmap(_sample_bucket))(bucket_lp_row, bkt_keys).astype(jnp.int32)
    bucket_idx = jnp.where(source_valid, bucket_idx, jnp.int32(0))

    bkt_lp = jnp.take_along_axis(bucket_lp_row, bucket_idx[..., None], axis=-1).squeeze(-1)
    bkt_lp = jnp.where(source_valid, bkt_lp, jnp.float32(0.0))

    log_prob = tgt_lp + bkt_lp

    return {
        "target_idx": target_idx,
        "bucket_idx": bucket_idx,
        "log_prob": log_prob,
        "entropy_target": entropy_target,
        "entropy_bucket": entropy_bucket,
        "source_valid": source_valid,
        "target_valid_any": target_valid_any,
    }


def pack_padded_actions(
    target_idx: jnp.ndarray,        # (B, P)
    bucket_idx: jnp.ndarray,        # (B, P)
    source_valid: jnp.ndarray,      # (B, P)
    action_grid: dict,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Pack per-source decisions into `(B, MAX_MOVES_PER_PLAYER, 3)` action tensors.

    Source planets without a valid action contribute mask=0 (zero row).
    Because MAX_PLANETS >= MAX_MOVES_PER_PLAYER could be violated (96 > 48),
    we *sort* sources by `source_valid` so all valid sources land in the
    first MAX_MOVES_PER_PLAYER slots, then truncate.
    """
    b, p = target_idx.shape
    from_ids = action_grid["from_ids"]                          # (B, P)
    angle_grid = action_grid["angle"]                           # (B, P, P, BUCKETS)
    ship_counts = action_grid["ship_counts"]                    # (B, P, P, BUCKETS)

    # Gather chosen angle/ship_count per source.
    s_range = jnp.arange(p, dtype=jnp.int32)
    b_range = jnp.arange(b, dtype=jnp.int32)
    bi, si = jnp.meshgrid(b_range, s_range, indexing="ij")
    angle = angle_grid[bi, si, target_idx, bucket_idx]          # (B, P)
    ships = ship_counts[bi, si, target_idx, bucket_idx]         # (B, P)

    # Build the per-source row.
    rows = jnp.stack([from_ids, angle, ships], axis=-1)         # (B, P, 3)

    # Identify NOOP moves (target == source)
    is_noop = (target_idx == s_range[None, :])                  # (B, P)

    # Mask is true only for valid sources that are NOT noops.
    # We zero out the action for NOOPs so no ships are launched.
    env_mask = source_valid & (~is_noop)
    mask = env_mask.astype(jnp.float32)                         # (B, P)
    rows = rows * mask[..., None]

    # Compact sources so all valid ones are first. We sort by NEGATIVE mask
    # (valid sources have key=-1 -> sort first), then truncate.
    sort_key = (-mask).astype(jnp.float32)
    sort_idx = jnp.argsort(sort_key, axis=-1)                   # (B, P)
    rows_sorted = jnp.take_along_axis(rows, sort_idx[..., None].repeat(3, axis=-1), axis=1)
    mask_sorted = jnp.take_along_axis(mask, sort_idx, axis=-1)

    actions = rows_sorted[:, :MAX_MOVES_PER_PLAYER, :]
    action_mask = mask_sorted[:, :MAX_MOVES_PER_PLAYER]

    # Truncation fix (Point 5): identify which sources actually "made the cut".
    # PPO should only learn from actions that were not truncated.
    rank = jnp.argsort(sort_idx, axis=-1)                       # (B, P) — rank of each source in sorted list
    executed_mask = source_valid & (rank < MAX_MOVES_PER_PLAYER) & (~is_noop)

    return actions, action_mask, executed_mask


def policy_step(
    rng: jax.Array,
    policy_apply,
    params,
    states,                          # vmapped (leading B) OrbitWarsState
    features: dict,                  # vmapped encoder output
    player_per_env: jnp.ndarray,     # (B,) int32 — which player this batch sees
    deterministic: bool = False,
) -> dict:
    """One full policy step:

    encode (done outside) -> policy forward -> sample masked actions -> pack.

    Returns a dict containing the action tensor + mask (ready for `step_jit`)
    and the per-row info PPO needs (`log_prob`, `entropy`, `value`,
    `executed_mask`).
    """
    out = policy_apply(params, **features)
    grid = jax.vmap(compose_action_grid, in_axes=(0, 0))(states, player_per_env)
    sampled = sample_actions(
        rng, out.target_logits, out.bucket_logits, grid, deterministic=deterministic
    )
    actions, action_mask, executed_mask = pack_padded_actions(
        sampled["target_idx"], sampled["bucket_idx"], sampled["source_valid"], grid
    )
    return {
        "actions": actions,                          # (B, MAX_MOVES, 3)
        "action_mask": action_mask,                  # (B, MAX_MOVES)
        "target_idx": sampled["target_idx"],         # (B, P) — for PPO
        "bucket_idx": sampled["bucket_idx"],
        "log_prob": sampled["log_prob"],
        "entropy_target": sampled["entropy_target"],
        "entropy_bucket": sampled["entropy_bucket"],
        "executed_mask": executed_mask,              # (B, P) — for PPO (fixes truncation bias)
        "value": out.value,                          # (B,)
    }


In [ ]:
%%writefile orbit_wars/state.py
"""Padded Orbit Wars game state for JAX simulation."""

from __future__ import annotations

from flax import struct
import jax.numpy as jnp

from .constants import (
    DEFAULT_EPISODE_STEPS,
    DEFAULT_SHIP_SPEED,
    FLEET_COLS,
    MAX_COMET_GROUPS,
    MAX_COMET_PATH_LEN,
    MAX_COMET_PLANETS,
    MAX_FLEETS,
    MAX_PLANETS,
    NUM_PLAYERS,
    PLANET_COLS,
)


@struct.dataclass
class CometGroups:
    active: jnp.ndarray  # (MAX_COMET_GROUPS,) bool
    planet_ids: jnp.ndarray  # (MAX_COMET_GROUPS, 4) int32
    path_index: jnp.ndarray  # (MAX_COMET_GROUPS,) int32
    paths: jnp.ndarray  # (MAX_COMET_GROUPS, 4, MAX_COMET_PATH_LEN, 2) float32
    path_lengths: jnp.ndarray  # (MAX_COMET_GROUPS, 4) int32


@struct.dataclass
class OrbitWarsState:
    planets: jnp.ndarray  # (MAX_PLANETS, PLANET_COLS)
    initial_planets: jnp.ndarray  # (MAX_PLANETS, PLANET_COLS)
    n_planets: jnp.int32
    fleets: jnp.ndarray  # (MAX_FLEETS, FLEET_COLS)
    n_fleets: jnp.int32
    comets: CometGroups
    comet_planet_ids: jnp.ndarray  # (MAX_COMET_PLANETS,) int32, -1 pad
    n_comet_planet_ids: jnp.int32
    angular_velocity: jnp.float32
    step: jnp.int32
    next_fleet_id: jnp.int32
    episode_seed: jnp.int32
    done: jnp.bool_
    rewards: jnp.ndarray  # (NUM_PLAYERS,) float32 terminal
    step_rewards: jnp.ndarray  # (NUM_PLAYERS,) float32 shaping
    ship_speed: jnp.float32
    episode_steps: jnp.int32


def empty_comet_groups() -> CometGroups:
    return CometGroups(
        active=jnp.zeros((MAX_COMET_GROUPS,), dtype=jnp.bool_),
        planet_ids=jnp.full((MAX_COMET_GROUPS, 4), -1, dtype=jnp.int32),
        path_index=jnp.full((MAX_COMET_GROUPS,), -1, dtype=jnp.int32),
        paths=jnp.zeros((MAX_COMET_GROUPS, 4, MAX_COMET_PATH_LEN, 2), dtype=jnp.float32),
        path_lengths=jnp.zeros((MAX_COMET_GROUPS, 4), dtype=jnp.int32),
    )


def empty_state() -> OrbitWarsState:
    return OrbitWarsState(
        planets=jnp.zeros((MAX_PLANETS, PLANET_COLS), dtype=jnp.float32),
        initial_planets=jnp.zeros((MAX_PLANETS, PLANET_COLS), dtype=jnp.float32),
        n_planets=jnp.int32(0),
        fleets=jnp.zeros((MAX_FLEETS, FLEET_COLS), dtype=jnp.float32),
        n_fleets=jnp.int32(0),
        comets=empty_comet_groups(),
        comet_planet_ids=jnp.full((MAX_COMET_PLANETS,), -1, dtype=jnp.int32),
        n_comet_planet_ids=jnp.int32(0),
        angular_velocity=jnp.float32(0.0),
        step=jnp.int32(0),
        next_fleet_id=jnp.int32(0),
        episode_seed=jnp.int32(0),
        done=jnp.bool_(False),
        rewards=jnp.zeros((NUM_PLAYERS,), dtype=jnp.float32),
        step_rewards=jnp.zeros((NUM_PLAYERS,), dtype=jnp.float32),
        ship_speed=jnp.float32(DEFAULT_SHIP_SPEED),
        episode_steps=jnp.int32(DEFAULT_EPISODE_STEPS),
    )


In [ ]:
%%writefile orbit_wars/step.py
"""JAX simulation step for Orbit Wars (vectorized, vmap-friendly)."""

from __future__ import annotations

import functools

import jax
import jax.numpy as jnp
import numpy as np

from .comet import spawn_comet_for_state
from .constants import (
    BOARD_SIZE,
    CENTER,
    COMET_SPAWN_STEPS,
    FLEET_COLS,
    MAX_COMET_GROUPS,
    MAX_COMET_PATH_LEN,
    MAX_COMET_PLANETS,
    MAX_FLEETS,
    MAX_MOVES_PER_PLAYER,
    MAX_PLANETS,
    NUM_PLAYERS,
    PLANET_COLS,
    ROTATION_RADIUS_LIMIT,
    SUN_RADIUS,
)
from .convert import pack_comets
from .geometry import fleet_speed, in_bounds, point_to_segment_distance, sun_hit, swept_pair_hit
from .state import CometGroups, OrbitWarsState

# ---------------------------------------------------------------------------
# Python-side comet bookkeeping (rare: at most ~10 expiries + 5 spawns per game)
# ---------------------------------------------------------------------------


def remove_expired_comets(state: OrbitWarsState) -> OrbitWarsState:
    """Mark comet planets inactive when their path is exhausted.

    Pure-Python numpy implementation; cheap because it short-circuits when no
    comet groups are active.
    """
    active_groups = np.asarray(state.comets.active)
    if not active_groups.any():
        return state

    planets = np.asarray(state.planets).copy()
    initial = np.asarray(state.initial_planets).copy()
    planet_ids = np.asarray(state.comets.planet_ids)
    path_index = np.asarray(state.comets.path_index)
    path_lengths = np.asarray(state.comets.path_lengths)

    # Build a map planet_id -> slot for active planets once.
    active_mask = planets[:, 7] > 0.0
    id_to_slot = {int(planets[i, 0]): i for i in range(MAX_PLANETS) if active_mask[i]}

    mutated = False
    for gi in range(MAX_COMET_GROUPS):
        if not active_groups[gi]:
            continue
        idx_next = int(path_index[gi]) + 1
        for pi in range(4):
            pid = int(planet_ids[gi, pi])
            plen = int(path_lengths[gi, pi])
            if pid < 0 or plen <= 0 or idx_next < plen:
                continue
            slot = id_to_slot.get(pid)
            if slot is None:
                continue
            planets[slot, 7] = 0.0
            initial[slot, 7] = 0.0
            mutated = True

    if not mutated:
        return state
    return state.replace(planets=jnp.asarray(planets), initial_planets=jnp.asarray(initial))


def _maybe_spawn_comet_numpy(state: OrbitWarsState) -> OrbitWarsState:
    next_step = int(state.step) + 1
    if next_step not in COMET_SPAWN_STEPS or bool(state.done):
        return state

    spawn = spawn_comet_for_state(
        np.asarray(state.planets),
        int(state.n_planets),
        np.asarray(state.initial_planets),
        np.asarray(state.comet_planet_ids),
        int(state.n_comet_planet_ids),
        float(state.angular_velocity),
        next_step,
        int(state.episode_seed),
        comet_speed=4.0,
    )
    if spawn is None:
        return state

    planets = np.asarray(state.planets).copy()
    initial = np.asarray(state.initial_planets).copy()
    n = int(state.n_planets)
    for row in spawn["new_planets"]:
        if n >= MAX_PLANETS:
            break
        planets[n, :7] = row
        planets[n, 7] = 1.0
        initial[n, :7] = row
        initial[n, 7] = 1.0
        n += 1

    comet_ids = np.asarray(state.comet_planet_ids).copy()
    nc = int(state.n_comet_planet_ids)
    for pid in spawn["new_comet_ids"]:
        if nc >= MAX_COMET_PLANETS:
            break
        comet_ids[nc] = int(pid)
        nc += 1

    from .convert import _unpack_comets

    comets_list = _unpack_comets(state.comets)
    comets_list.append(spawn["group"])
    comets = pack_comets(comets_list)

    return state.replace(
        planets=jnp.asarray(planets),
        initial_planets=jnp.asarray(initial),
        n_planets=jnp.int32(n),
        comets=comets,
        comet_planet_ids=jnp.asarray(comet_ids),
        n_comet_planet_ids=jnp.int32(nc),
    )


# ---------------------------------------------------------------------------
# Vectorized JIT physics
# ---------------------------------------------------------------------------


def _apply_moves(
    state: OrbitWarsState,
    actions: jnp.ndarray,
    action_mask: jnp.ndarray,
    player: jnp.int32,
) -> OrbitWarsState:
    """Apply one player's moves sequentially.

    Sequentiality only matters for ship deduction (subsequent moves from the
    same source see the deducted balance). We precompute source-planet slots
    once (vectorized) and only run the small sequential scan over moves.
    """
    planets = state.planets
    fleets = state.fleets
    n_fleets = state.n_fleets
    next_fleet_id = state.next_fleet_id

    pids_i32 = planets[:, 0].astype(jnp.int32)
    active_planet = planets[:, 7] > 0.0

    move_from = actions[:, 0].astype(jnp.int32)              # (M,)
    move_angle = actions[:, 1]                                # (M,)
    move_ships_i32 = actions[:, 2].astype(jnp.int32)          # (M,)

    match = (move_from[:, None] == pids_i32[None, :]) & active_planet[None, :]   # (M, P)
    source_idx = jnp.argmax(match.astype(jnp.int32), axis=-1)                    # (M,)
    source_exists = jnp.any(match, axis=-1)                                      # (M,)

    use_move = (action_mask > 0.0) & (move_ships_i32 > 0) & source_exists        # (M,)

    player_f = player.astype(jnp.float32)

    def body(i, carry):
        planets_c, fleets_c, n_fleets_c, next_id_c = carry
        sidx = source_idx[i]
        safe_sidx = jnp.maximum(sidx, 0)
        owner = planets_c[safe_sidx, 1]
        have = planets_c[safe_sidx, 5].astype(jnp.int32)
        ships = move_ships_i32[i]
        valid = use_move[i] & (owner == player_f) & (have >= ships)

        new_ship_count = planets_c[safe_sidx, 5] - ships.astype(jnp.float32)
        planets_c = planets_c.at[safe_sidx, 5].set(
            jnp.where(valid, new_ship_count, planets_c[safe_sidx, 5])
        )
        radius = planets_c[safe_sidx, 4]
        start_x = planets_c[safe_sidx, 2] + jnp.cos(move_angle[i]) * (radius + 0.1)
        start_y = planets_c[safe_sidx, 3] + jnp.sin(move_angle[i]) * (radius + 0.1)

        slot = n_fleets_c
        can_add = valid & (slot < MAX_FLEETS)
        safe_slot = jnp.minimum(slot, MAX_FLEETS - 1)
        new_row = jnp.array(
            [
                next_id_c.astype(jnp.float32),
                player_f,
                start_x,
                start_y,
                move_angle[i],
                move_from[i].astype(jnp.float32),
                ships.astype(jnp.float32),
                1.0,
            ],
            dtype=jnp.float32,
        )
        fleets_c = fleets_c.at[safe_slot].set(
            jnp.where(can_add, new_row, fleets_c[safe_slot])
        )
        n_fleets_c = jnp.where(can_add, n_fleets_c + 1, n_fleets_c)
        next_id_c = jnp.where(can_add, next_id_c + 1, next_id_c)
        return (planets_c, fleets_c, n_fleets_c, next_id_c)

    planets, fleets, n_fleets, next_fleet_id = jax.lax.fori_loop(
        0, MAX_MOVES_PER_PLAYER, body, (planets, fleets, n_fleets, next_fleet_id),
    )
    return state.replace(
        planets=planets, fleets=fleets, n_fleets=n_fleets, next_fleet_id=next_fleet_id,
    )


def _production(state: OrbitWarsState) -> OrbitWarsState:
    owned = (state.planets[:, 1] >= 0.0) & (state.planets[:, 7] > 0.0)
    planets = state.planets.at[:, 5].set(
        jnp.where(owned, state.planets[:, 5] + state.planets[:, 6], state.planets[:, 5])
    )
    return state.replace(planets=planets)


def _compute_planet_paths(state: OrbitWarsState):
    """Returns (pids, old_x, old_y, new_x, new_y, radius, check).

    Vectorized: no per-planet fori_loop. Comet positions are computed for each
    of the (MAX_COMET_GROUPS, 4) comet planet slots and scattered into the
    planet array by slot index.
    """
    pids_i32 = state.planets[:, 0].astype(jnp.int32)
    old_x = state.planets[:, 2]
    old_y = state.planets[:, 3]
    radius = state.planets[:, 4]
    active = state.planets[:, 7] > 0.0

    # is_comet via broadcast (P, K)
    cpids = state.comet_planet_ids  # (MAX_COMET_PLANETS,) int32, -1 padded
    valid_cpid = cpids >= 0
    is_comet = active & jnp.any(
        (pids_i32[:, None] == cpids[None, :]) & valid_cpid[None, :], axis=-1
    )

    # Rotation for non-comet active planets
    init = state.initial_planets
    dx0 = init[:, 2] - CENTER
    dy0 = init[:, 3] - CENTER
    orbit_r = jnp.sqrt(dx0 * dx0 + dy0 * dy0)
    rotating = active & (orbit_r + radius < ROTATION_RADIUS_LIMIT) & (~is_comet)
    initial_angle = jnp.arctan2(dy0, dx0)
    current_angle = initial_angle + state.angular_velocity * state.step.astype(jnp.float32)
    rot_x = CENTER + orbit_r * jnp.cos(current_angle)
    rot_y = CENTER + orbit_r * jnp.sin(current_angle)
    new_x = jnp.where(rotating, rot_x, old_x)
    new_y = jnp.where(rotating, rot_y, old_y)
    check = jnp.where(active & (~is_comet), 1.0, 0.0)

    # ---- Comet path lookup (vectorized scatter) ------------------------
    comets = state.comets
    cgpids = comets.planet_ids                       # (G, 4) int32
    cplens = comets.path_lengths                     # (G, 4) int32
    cactive = comets.active                          # (G,) bool
    idx_g = comets.path_index + 1                    # (G,) int32
    idx_clip = jnp.clip(idx_g, 0, MAX_COMET_PATH_LEN - 1)

    # For each (g, q) find the matching planet slot. Many of these will be
    # padding (cgpids < 0) — we mask those out.
    match_gqp = (
        (cgpids[..., None] == pids_i32[None, None, :])
        & active[None, None, :]
        & (cgpids[..., None] >= 0)
    )                                                # (G, 4, P)
    slot_gq = jnp.argmax(match_gqp.astype(jnp.int32), axis=-1)  # (G, 4)
    has_match_gq = jnp.any(match_gqp, axis=-1)                  # (G, 4)

    on_board_gq = (idx_g[:, None] < cplens) & has_match_gq & cactive[:, None]
    is_comet_slot_gq = has_match_gq & cactive[:, None]

    # Gather positions: paths[g, q, idx_g[g], :]
    g_arange = jnp.arange(MAX_COMET_GROUPS)
    q_arange = jnp.arange(4)
    g_grid, q_grid = jnp.meshgrid(g_arange, q_arange, indexing="ij")  # (G, 4)
    idx_grid = jnp.broadcast_to(idx_clip[:, None], (MAX_COMET_GROUPS, 4))
    path_xy = comets.paths[g_grid, q_grid, idx_grid, :]  # (G, 4, 2)
    path_px = path_xy[..., 0]
    path_py = path_xy[..., 1]

    # Flatten to (G*4,) for scatter.
    flat_slot = slot_gq.reshape(-1)
    flat_on = on_board_gq.reshape(-1)
    flat_px = path_px.reshape(-1)
    flat_py = path_py.reshape(-1)
    flat_is_comet_slot = is_comet_slot_gq.reshape(-1)

    # Scatter on-board comet positions. Multiple writes to the same slot
    # cannot happen because each planet id is unique within active groups.
    nx_at_slot = jnp.where(flat_on, flat_px, new_x[flat_slot])
    ny_at_slot = jnp.where(flat_on, flat_py, new_y[flat_slot])
    new_x = new_x.at[flat_slot].set(nx_at_slot)
    new_y = new_y.at[flat_slot].set(ny_at_slot)

    # Comet planet slots get check=1 (collision-eligible) whether on or off
    # board, mirroring the reference env behaviour.
    check_at_slot = jnp.where(flat_is_comet_slot, 1.0, check[flat_slot])
    check = check.at[flat_slot].set(check_at_slot)

    return pids_i32, old_x, old_y, new_x, new_y, radius, check


def _move_fleets(state, pids_i32, old_x, old_y, new_x, new_y, radius, check):
    """Returns (state, combat[planet, player]).

    Vectorized: builds the full (F, P) swept-collision matrix in one op.
    """
    max_speed = state.ship_speed
    fleets = state.fleets

    active_f = fleets[:, 7] > 0.0                       # (F,)
    owner = fleets[:, 1].astype(jnp.int32)              # (F,)
    angle = fleets[:, 4]
    ships = fleets[:, 6]
    old_fx = fleets[:, 2]
    old_fy = fleets[:, 3]
    speed = fleet_speed(ships, max_speed)
    new_fx = old_fx + jnp.cos(angle) * speed
    new_fy = old_fy + jnp.sin(angle) * speed

    # (F, P) swept-collision matrix
    hit_fp = swept_pair_hit(
        old_fx[:, None], old_fy[:, None], new_fx[:, None], new_fy[:, None],
        old_x[None, :], old_y[None, :], new_x[None, :], new_y[None, :], radius[None, :],
    )                                                   # (F, P)
    eligible = (check[None, :] > 0.0) & active_f[:, None]
    hit_fp = hit_fp & eligible

    # First-hit planet per fleet (lowest planet index that hits)
    any_hit = jnp.any(hit_fp, axis=-1)                  # (F,)
    first_hit = jnp.argmax(hit_fp.astype(jnp.int32), axis=-1)  # (F,)
    hit_idx = jnp.where(any_hit, first_hit, -1)

    # Sun + bounds
    out_bounds = ~in_bounds(new_fx, new_fy)
    sun = sun_hit(old_fx, old_fy, new_fx, new_fy)
    remove = active_f & ((hit_idx >= 0) | out_bounds | sun)

    # Scatter-add ships into combat[planet, player]
    combat = jnp.zeros((MAX_PLANETS, NUM_PLAYERS), dtype=jnp.float32)
    hit_mask = active_f & (hit_idx >= 0)
    safe_hit_idx = jnp.where(hit_mask, hit_idx, 0)
    safe_owner = jnp.clip(owner, 0, NUM_PLAYERS - 1)
    contrib = jnp.where(hit_mask, ships, 0.0)
    combat = combat.at[safe_hit_idx, safe_owner].add(contrib)

    fleets = fleets.at[:, 2].set(new_fx)
    fleets = fleets.at[:, 3].set(new_fy)
    fleets = fleets.at[:, 7].set(jnp.where(remove, 0.0, fleets[:, 7]))
    return state.replace(fleets=fleets), combat


def _apply_planet_positions(state, new_x, new_y) -> OrbitWarsState:
    planets = state.planets.at[:, 2].set(new_x)
    planets = planets.at[:, 3].set(new_y)
    return state.replace(planets=planets)


def _advance_comet_indices(state: OrbitWarsState) -> OrbitWarsState:
    comets = state.comets
    comets = comets.replace(path_index=comets.path_index + 1)
    return state.replace(comets=comets)


def _expire_comets_in_jit(state: OrbitWarsState) -> OrbitWarsState:
    """Deactivate comet planets whose path has ended (vectorized, JIT-safe).

    After `_advance_comet_indices`, comets.path_index points at the freshly
    consumed step. A comet quad has expired when path_index >= path_length
    (i.e. we have moved past the last waypoint). Mirrors the Python
    `remove_expired_comets` behaviour.
    """
    comets = state.comets
    cgpids = comets.planet_ids                       # (G, 4)
    cplens = comets.path_lengths                     # (G, 4)
    cactive = comets.active                          # (G,)
    idx_now = comets.path_index                      # (G,) — already advanced

    pids_i32 = state.planets[:, 0].astype(jnp.int32)
    active = state.planets[:, 7] > 0.0

    expired_gq = (idx_now[:, None] >= cplens) & (cplens > 0) & cactive[:, None] & (cgpids >= 0)
    # For each (g, q) where expired, find the matching planet slot and clear active.
    match_gqp = (
        (cgpids[..., None] == pids_i32[None, None, :])
        & active[None, None, :]
        & (cgpids[..., None] >= 0)
    )
    slot_gq = jnp.argmax(match_gqp.astype(jnp.int32), axis=-1)
    has_match_gq = jnp.any(match_gqp, axis=-1)
    do_expire = (expired_gq & has_match_gq).reshape(-1)
    flat_slot = slot_gq.reshape(-1)

    planets = state.planets
    new_active = jnp.where(do_expire, 0.0, planets[flat_slot, 7])
    planets = planets.at[flat_slot, 7].set(new_active)
    initial = state.initial_planets
    new_active_i = jnp.where(do_expire, 0.0, initial[flat_slot, 7])
    initial = initial.at[flat_slot, 7].set(new_active_i)
    return state.replace(planets=planets, initial_planets=initial)


def _resolve_combat(state: OrbitWarsState, combat: jnp.ndarray) -> OrbitWarsState:
    """Pure elementwise combat resolution (no fori_loop)."""
    planets = state.planets
    active = planets[:, 7] > 0.0
    ships_p0 = combat[:, 0]
    ships_p1 = combat[:, 1]
    total = ships_p0 + ships_p1
    contested = active & (total > 0.0)

    top = jnp.maximum(ships_p0, ships_p1)
    second = jnp.minimum(ships_p0, ships_p1)
    tie = ships_p0 == ships_p1
    survivor_ships = jnp.where(tie, 0.0, top - second)
    top_player = jnp.where(ships_p0 >= ships_p1, 0, 1).astype(jnp.int32)
    survivor_owner = jnp.where(survivor_ships > 0.0, top_player, -1)

    owner = planets[:, 1].astype(jnp.int32)
    same = owner == survivor_owner
    diff = (~same) & (survivor_owner >= 0)

    new_ships_same = planets[:, 5] + survivor_ships
    new_ships_diff = planets[:, 5] - survivor_ships
    captured = new_ships_diff < 0.0
    new_ships = jnp.where(
        same,
        new_ships_same,
        jnp.where(captured, jnp.abs(new_ships_diff), new_ships_diff),
    )
    new_owner = jnp.where(
        contested & diff & captured,
        survivor_owner.astype(jnp.float32),
        planets[:, 1],
    )
    planets = planets.at[:, 5].set(jnp.where(contested, new_ships, planets[:, 5]))
    planets = planets.at[:, 1].set(jnp.where(contested & diff, new_owner, planets[:, 1]))
    return state.replace(planets=planets)


def _termination(state: OrbitWarsState) -> OrbitWarsState:
    step = state.step
    terminated_by_steps = step >= (state.episode_steps - 2)

    owned = (state.planets[:, 1] >= 0.0) & (state.planets[:, 7] > 0.0)
    fleet_active = state.fleets[:, 7] > 0.0
    fleet_owners = state.fleets[:, 1].astype(jnp.int32)
    planet_owners = state.planets[:, 1].astype(jnp.int32)

    # Vectorized presence + score per player (NUM_PLAYERS == 2).
    p_range = jnp.arange(NUM_PLAYERS, dtype=jnp.int32)
    owned_match = owned[None, :] & (planet_owners[None, :] == p_range[:, None])     # (P_n, P)
    fleet_match = fleet_active[None, :] & (fleet_owners[None, :] == p_range[:, None])  # (P_n, F)

    has_any = jnp.any(owned_match, axis=-1) | jnp.any(fleet_match, axis=-1)
    alive_count = jnp.sum(has_any.astype(jnp.int32))
    terminated = terminated_by_steps | (alive_count <= 1)

    planet_score = jnp.sum(
        jnp.where(owned_match, state.planets[None, :, 5], 0.0), axis=-1
    )
    fleet_score = jnp.sum(
        jnp.where(fleet_match, state.fleets[None, :, 6], 0.0), axis=-1
    )
    scores = planet_score + fleet_score                          # (NUM_PLAYERS,)

    max_score = jnp.max(scores)
    all_max = jnp.all(scores == max_score)
    
    # Reward: 1.0 for strictly winning, -1.0 for strictly losing, 0.0 for ties.
    rewards = jnp.where(
        terminated,
        jnp.where(
            all_max | (max_score <= 0.0),
            jnp.zeros((NUM_PLAYERS,), dtype=jnp.float32),
            jnp.where(scores == max_score, 1.0, -1.0)
        ),
        jnp.zeros((NUM_PLAYERS,), dtype=jnp.float32),
    )

    # Intermediate reward shaping: tiny bonus per planet owned.
    # Encourages early game expansion.
    step_rewards = jnp.sum(owned_match.astype(jnp.float32), axis=-1) * 0.01

    return state.replace(done=terminated, rewards=rewards, step_rewards=step_rewards)


# ---------------------------------------------------------------------------
# Public step API
# ---------------------------------------------------------------------------


@jax.jit
def step_jit(
    state: OrbitWarsState,
    actions_p0: jnp.ndarray,
    actions_p1: jnp.ndarray,
    mask_p0: jnp.ndarray,
    mask_p1: jnp.ndarray,
) -> OrbitWarsState:
    state = _apply_moves(state, actions_p0, mask_p0, jnp.int32(0))
    state = _apply_moves(state, actions_p1, mask_p1, jnp.int32(1))
    state = _production(state)
    pids, old_x, old_y, new_x, new_y, radius, check = _compute_planet_paths(state)
    state, combat = _move_fleets(state, pids, old_x, old_y, new_x, new_y, radius, check)
    state = _apply_planet_positions(state, new_x, new_y)
    state = _advance_comet_indices(state)
    state = _expire_comets_in_jit(state)
    state = _resolve_combat(state, combat)
    state = state.replace(step=state.step + 1)
    state = _termination(state)
    return state


def step(
    state: OrbitWarsState,
    actions: list[list[list[float | int]]] | None = None,
    *,
    actions_p0: jnp.ndarray | None = None,
    actions_p1: jnp.ndarray | None = None,
    mask_p0: jnp.ndarray | None = None,
    mask_p1: jnp.ndarray | None = None,
) -> OrbitWarsState:
    """Full step: Python-side comet spawn (rare) then vectorized JIT physics.

    Comet *expiry* now happens inside `step_jit` (mask-based). This function
    only handles comet *spawning*, which still depends on RNG/path generation
    that lives in numpy.
    """
    state = _maybe_spawn_comet_numpy(state)

    if actions is not None:
        a0, m0 = _list_action_to_padded(actions[0])
        a1, m1 = _list_action_to_padded(actions[1])
    else:
        a0, m0 = actions_p0, mask_p0
        a1, m1 = actions_p1, mask_p1
    assert a0 is not None and a1 is not None and m0 is not None and m1 is not None

    state = step_jit(state, a0, a1, m0, m1)
    return state


def _list_action_to_padded(moves: list[list[float | int]]) -> tuple[jnp.ndarray, jnp.ndarray]:
    arr = np.zeros((MAX_MOVES_PER_PLAYER, 3), dtype=np.float32)
    mask = np.zeros((MAX_MOVES_PER_PLAYER,), dtype=np.float32)
    n = min(len(moves), MAX_MOVES_PER_PLAYER)
    for i in range(n):
        move = moves[i]
        if len(move) != 3:
            continue
        arr[i, 0] = float(move[0])
        arr[i, 1] = float(move[1])
        arr[i, 2] = float(move[2])
        mask[i] = 1.0
    return jnp.asarray(arr), jnp.asarray(mask)


@jax.jit
def batched_step(
    states: OrbitWarsState,
    actions_p0: jnp.ndarray,
    actions_p1: jnp.ndarray,
    mask_p0: jnp.ndarray,
    mask_p1: jnp.ndarray,
) -> OrbitWarsState:
    """Vectorized batched step. Requires pre-baked comets (no Python spawn)."""
    return jax.vmap(step_jit)(states, actions_p0, actions_p1, mask_p0, mask_p1)


In [ ]:
%%writefile policy.py
"""Transformer policy for Orbit Wars (JAX/Flax).

Consumes the per-entity feature dict produced by
`orbit_wars.features_jax.encode_observation` and emits:

- target logits over MAX_PLANETS target slots (which planet to attack/reinforce)
- ship-bucket logits over (MAX_PLANETS, BUCKET_COUNT) conditional on target
- a single scalar value (for the critic)

Design (Expert-optimized):

    tokens = [global_token, planet_tokens, fleet_tokens]
    tokens = TransformerEncoder(d_model, n_heads, n_layers)(tokens, mask)
    target_logits = dot_product(planet_h, planet_h) + noop_bias
    bucket_logits = MLP(planet_h_src, planet_h_tgt)
    value         = MLP(global_h)
"""

from __future__ import annotations

import jax
import jax.numpy as jnp
from flax import linen as nn
from flax import struct


@struct.dataclass
class PolicyOutput:
    target_logits: jnp.ndarray   # (B, MAX_PLANETS, MAX_PLANETS)
    bucket_logits: jnp.ndarray   # (B, MAX_PLANETS, MAX_PLANETS, BUCKET_COUNT)
    value: jnp.ndarray           # (B,)


class TransformerBlock(nn.Module):
    d_model: int
    num_heads: int
    ff_mult: int = 4

    @nn.compact
    def __call__(self, tokens: jnp.ndarray, kv_padding_mask: jnp.ndarray) -> jnp.ndarray:
        # tokens: (B, T, d)
        # kv_padding_mask: (B, T) bool — True for *valid* tokens.
        b, t, _ = tokens.shape

        mask = kv_padding_mask[:, None, None, :]                      # (B, 1, 1, T)
        mask = jnp.broadcast_to(mask, (b, self.num_heads, t, t))

        attn = nn.MultiHeadDotProductAttention(
            num_heads=self.num_heads,
            qkv_features=self.d_model,
            out_features=self.d_model,
            kernel_init=nn.initializers.xavier_uniform(),
        )
        x_norm = nn.LayerNorm()(tokens)
        x_attn = attn(x_norm, x_norm, mask=mask)
        tokens = tokens + x_attn

        y_norm = nn.LayerNorm()(tokens)
        y = nn.Dense(self.d_model * self.ff_mult)(y_norm)
        y = nn.gelu(y)
        y = nn.Dense(self.d_model)(y)
        tokens = tokens + y

        tokens = tokens * kv_padding_mask[:, :, None].astype(tokens.dtype)
        return tokens


class PlanetPolicy(nn.Module):
    """Transformer over (global, planets, fleets) producing planet-action heads."""

    planet_count: int
    fleet_count: int
    bucket_count: int = 8
    d_model: int = 96
    num_heads: int = 4
    num_layers: int = 3
    ff_mult: int = 4
    noop_bias_init: float = 2.0

    def setup(self) -> None:
        self.global_in = nn.Dense(self.d_model)
        self.planet_in = nn.Dense(self.d_model)
        self.fleet_in = nn.Dense(self.d_model)
        
        # Token-type embeddings: 0=Global, 1=Planet, 2=Fleet
        self.type_emb = nn.Embed(3, self.d_model)
        
        self.blocks = [
            TransformerBlock(self.d_model, self.num_heads, self.ff_mult)
            for _ in range(self.num_layers)
        ]
        
        self.target_proj_q = nn.Dense(self.d_model)
        self.target_proj_k = nn.Dense(self.d_model)
        
        # Bucket head takes concatenated source and target planet representations
        self.bucket_head = nn.Sequential([
            nn.Dense(self.d_model),
            nn.gelu,
            nn.Dense(self.bucket_count)
        ])
        
        self.value_head = nn.Sequential([
            nn.Dense(self.d_model),
            nn.gelu,
            nn.Dense(1)
        ])
        
        self.noop_bias = self.param(
            "noop_bias", nn.initializers.constant(self.noop_bias_init), ()
        )

    def __call__(
        self,
        planet_features: jnp.ndarray,    # (B, P, F_p)
        planet_mask: jnp.ndarray,        # (B, P) bool
        fleet_features: jnp.ndarray,     # (B, F, F_f)
        fleet_mask: jnp.ndarray,         # (B, F) bool
        global_features: jnp.ndarray,    # (B, F_g)
    ) -> PolicyOutput:
        b, p, _ = planet_features.shape
        f = fleet_features.shape[1]

        # 1. Project to d_model
        g_tok = self.global_in(global_features)[:, None, :]  # (B, 1, d)
        p_tok = self.planet_in(planet_features)              # (B, P, d)
        f_tok = self.fleet_in(fleet_features)                # (B, F, d)
        
        # 2. Add Type Embeddings
        type_idx = jnp.concatenate([
            jnp.zeros((1,), dtype=jnp.int32),
            jnp.ones((p,), dtype=jnp.int32),
            jnp.full((f,), 2, dtype=jnp.int32),
        ])
        t_embs = self.type_emb(type_idx)                     # (1+P+F, d)
        
        tokens = jnp.concatenate([g_tok, p_tok, f_tok], axis=1) # (B, 1+P+F, d)
        tokens = tokens + t_embs[None, :, :]
        
        full_mask = jnp.concatenate([
            jnp.ones((b, 1), dtype=jnp.bool_),
            planet_mask,
            fleet_mask
        ], axis=1)                                           # (B, 1+P+F)

        # 3. Transformer
        for block in self.blocks:
            tokens = block(tokens, full_mask)

        # 4. Extract Planet representations
        g_h = tokens[:, 0, :]                                # (B, d)
        p_h = tokens[:, 1 : 1 + p, :]                        # (B, P, d)

        # 5. Target Head (einsum bilinear)
        q = self.target_proj_q(p_h)                          # (B, P, d)
        k = self.target_proj_k(p_h)                          # (B, P, d)
        scale = jnp.float32(1.0 / jnp.sqrt(self.d_model))
        target_logits = jnp.einsum("bsd,btd->bst", q, k) * scale     # (B, P, P)

        # Apply NOOP bias to diagonal
        diag = jnp.eye(p, dtype=target_logits.dtype)
        target_logits = target_logits + diag[None, :, :] * self.noop_bias

        # 6. Bucket Head (conditioned on source AND target)
        h_src = p_h[:, :, None, :]                           # (B, P, 1, d)
        h_tgt = p_h[:, None, :, :]                           # (B, 1, P, d)
        pair_h = jnp.concatenate([
            jnp.broadcast_to(h_src, (b, p, p, self.d_model)),
            jnp.broadcast_to(h_tgt, (b, p, p, self.d_model))
        ], axis=-1)                                          # (B, P, P, 2d)
        bucket_logits = self.bucket_head(pair_h)             # (B, P, P, BUCKETS)

        # 7. Value Head (from Global token)
        value = self.value_head(g_h).squeeze(-1)             # (B,)

        return PolicyOutput(
            target_logits=target_logits,
            bucket_logits=bucket_logits,
            value=value,
        )


def init_policy(
    rng: jax.Array,
    model: PlanetPolicy,
    example: dict[str, jnp.ndarray],
):
    """Initialize params from an example batch dict."""
    return model.init(rng, **example)


In [ ]:
%%writefile ppo.py
"""PPO loss + GAE for the Transformer Orbit Wars policy.

All operations consume the per-source decision rows produced by
`orbit_wars.rollout.policy_step`. A "row" = one (env, time, source_planet)
triple. Rows where `source_valid == False` are masked out throughout.
"""

from __future__ import annotations

import jax
import jax.numpy as jnp

from orbit_wars.decode import BUCKET_COUNT, compose_action_grid


# ---------------------------------------------------------------------------
# GAE
# ---------------------------------------------------------------------------


def compute_gae(
    rewards: jnp.ndarray,           # (B, T)
    values: jnp.ndarray,             # (B, T)
    dones: jnp.ndarray,              # (B, T) — True if episode ended at this step
    next_value: jnp.ndarray,         # (B,) — value bootstrap after last step
    gamma: float,
    lam: float,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Generalized Advantage Estimation per env.

    Returns `(advantages, returns)` each shaped (B, T).

    Episode resets: when `dones[b, t] == True`, the bootstrap is zeroed at
    that step so advantages don't leak across episode boundaries.
    """
    not_done = 1.0 - dones.astype(jnp.float32)

    def scan_body(carry, x):
        next_v, next_gae = carry
        value, reward, nd = x
        delta = reward + gamma * next_v * nd - value
        gae = delta + gamma * lam * nd * next_gae
        return (value, gae), gae

    # Iterate in reverse over time (axis=1).
    rewards_t = jnp.transpose(rewards, (1, 0))         # (T, B)
    values_t = jnp.transpose(values, (1, 0))
    nd_t = jnp.transpose(not_done, (1, 0))

    init = (next_value, jnp.zeros_like(next_value))
    (_v, _gae), advs = jax.lax.scan(
        scan_body, init, (values_t, rewards_t, nd_t), reverse=True
    )
    advs = jnp.transpose(advs, (1, 0))                  # (B, T)
    returns = advs + values
    return advs, returns


# ---------------------------------------------------------------------------
# Joint log-prob (target + bucket) under a frozen action grid
# ---------------------------------------------------------------------------


_NEG_INF = jnp.float32(-1e9)


def _masked_log_softmax(logits: jnp.ndarray, mask: jnp.ndarray) -> jnp.ndarray:
    safe = jnp.where(mask, logits, _NEG_INF)
    any_valid = jnp.any(mask, axis=-1, keepdims=True)
    safe = jnp.where(any_valid, safe, jnp.zeros_like(logits))
    return jax.nn.log_softmax(safe, axis=-1)


def joint_log_prob_and_entropy(
    target_logits: jnp.ndarray,         # (N, P, P)
    bucket_logits: jnp.ndarray,         # (N, P, P, BUCKETS)
    target_has_bucket: jnp.ndarray,     # (N, P, P) bool
    bucket_valid: jnp.ndarray,          # (N, P, P, BUCKETS) bool
    target_idx: jnp.ndarray,            # (N, P) int32
    bucket_idx: jnp.ndarray,            # (N, P) int32
    executed_mask: jnp.ndarray,         # (N, P) bool
) -> dict[str, jnp.ndarray]:
    """Return joint log_prob, per-row entropy contributions, all masked by
    `executed_mask` so PPO can flatten and average."""
    tgt_lp = _masked_log_softmax(target_logits, target_has_bucket)        # (N, P, P)
    tgt_lp_sel = jnp.take_along_axis(tgt_lp, target_idx[..., None], axis=-1).squeeze(-1)
    tgt_lp_sel = jnp.where(executed_mask, tgt_lp_sel, 0.0)

    # Per-source entropy of the target distribution (sum over targets).
    tgt_p = jnp.exp(tgt_lp) * target_has_bucket.astype(tgt_lp.dtype)
    entropy_target = -jnp.sum(tgt_p * tgt_lp, axis=-1)                    # (N, P)
    entropy_target = jnp.where(executed_mask, entropy_target, 0.0)

    # Gather bucket logits and validity for the *chosen* target.
    # bucket_logits: (N, P, P, BUCKETS) -> (N, P, BUCKETS)
    b_idx = jnp.arange(target_idx.shape[0])[:, None]
    p_idx = jnp.arange(target_idx.shape[1])[None, :]
    chosen_bucket_logits = bucket_logits[b_idx, p_idx, target_idx]

    chosen_bucket_valid = jnp.take_along_axis(
        bucket_valid,
        target_idx[..., None, None].repeat(BUCKET_COUNT, axis=-1),
        axis=2,
    ).squeeze(2)                                                          # (N, P, BUCKETS)
    bkt_lp = _masked_log_softmax(chosen_bucket_logits, chosen_bucket_valid)
    bkt_lp_sel = jnp.take_along_axis(bkt_lp, bucket_idx[..., None], axis=-1).squeeze(-1)
    bkt_lp_sel = jnp.where(executed_mask, bkt_lp_sel, 0.0)

    bkt_p = jnp.exp(bkt_lp) * chosen_bucket_valid.astype(bkt_lp.dtype)
    entropy_bucket = -jnp.sum(bkt_p * bkt_lp, axis=-1)
    entropy_bucket = jnp.where(executed_mask, entropy_bucket, 0.0)

    return {
        "log_prob": tgt_lp_sel + bkt_lp_sel,
        "entropy_target": entropy_target,
        "entropy_bucket": entropy_bucket,
    }


# ---------------------------------------------------------------------------
# PPO loss
# ---------------------------------------------------------------------------


def ppo_loss_fn(
    params,
    apply_fn,
    batch: dict,
    clip_coef: float,
    vf_coef: float,
    ent_coef: float,
) -> tuple[jnp.ndarray, dict]:
    """Compute PPO clipped-objective loss + diagnostic metrics.

    Shapes:
        N = batch rows (env, time) flattened
        P = MAX_PLANETS source decisions per row

    `batch` must contain:
        planet_features (N, P, F_p), planet_mask (N, P),
        fleet_features  (N, F, F_f), fleet_mask  (N, F),
        global_features (N, F_g)
            — encoder outputs (recomputed at minibatch time).
        target_idx, bucket_idx, executed_mask, old_log_prob   (N, P)
            — chosen actions and per-source validity from rollout time.
        target_has_bucket (N, P, P) bool, bucket_valid (N, P, P, BUCKETS) bool
            — frozen action-grid masks captured at rollout time.
        advantages (N,), returns (N,)
            — GAE outputs at env-time granularity; broadcast over sources.
    """
    out = apply_fn(
        params,
        planet_features=batch["planet_features"],
        planet_mask=batch["planet_mask"],
        fleet_features=batch["fleet_features"],
        fleet_mask=batch["fleet_mask"],
        global_features=batch["global_features"],
    )                                                # value (N,), target_logits (N,P,P), bucket_logits (N,P,P,B)

    info = joint_log_prob_and_entropy(
        target_logits=out.target_logits,
        bucket_logits=out.bucket_logits,
        target_has_bucket=batch["target_has_bucket"],
        bucket_valid=batch["bucket_valid"],
        target_idx=batch["target_idx"],
        bucket_idx=batch["bucket_idx"],
        executed_mask=batch["executed_mask"],
    )
    new_log_prob = info["log_prob"]                  # (N, P)
    entropy = info["entropy_target"] + info["entropy_bucket"]   # (N, P)

    old_log_prob = batch["old_log_prob"]             # (N, P)
    adv_env = batch["advantages"]                    # (N,)
    returns_env = batch["returns"]                   # (N,)
    executed_mask = batch["executed_mask"]           # (N, P)
    mask_f = executed_mask.astype(jnp.float32)
    mask_count = jnp.maximum(jnp.sum(mask_f), 1.0)

    adv = adv_env[:, None]                           # (N, 1) — broadcast across sources
    ratio = jnp.exp(new_log_prob - old_log_prob)
    unclipped = ratio * adv
    clipped = jnp.clip(ratio, 1.0 - clip_coef, 1.0 + clip_coef) * adv
    policy_loss = -jnp.sum(jnp.minimum(unclipped, clipped) * mask_f) / mask_count

    # Critic: one scalar per env-time row.
    value_pred = out.value                           # (N,)
    value_loss = jnp.mean((returns_env - value_pred) ** 2)

    entropy_mean = jnp.sum(entropy * mask_f) / mask_count

    total_loss = policy_loss + vf_coef * value_loss - ent_coef * entropy_mean

    log_ratio = new_log_prob - old_log_prob
    approx_kl = jnp.sum((ratio - 1.0 - log_ratio) * mask_f) / mask_count
    clip_frac = jnp.sum(((jnp.abs(ratio - 1.0) > clip_coef).astype(jnp.float32)) * mask_f) / mask_count

    return total_loss, {
        "policy_loss": policy_loss,
        "value_loss": value_loss,
        "entropy": entropy_mean,
        "approx_kl": approx_kl,
        "clip_fraction": clip_frac,
    }


def explained_variance(returns: jnp.ndarray, values: jnp.ndarray) -> jnp.ndarray:
    """1 - Var(returns - values) / Var(returns). Returns 0 if returns is constant."""
    var_r = jnp.var(returns)
    return jnp.where(var_r < 1e-8, jnp.float32(0.0), 1.0 - jnp.var(returns - values) / var_r)


In [ ]:
%%writefile train_ppo.py
"""PPO trainer for the JAX Transformer Orbit Wars policy.

End-to-end design:

- `num_envs` envs are stacked via `tree_map(jnp.stack)` and stepped with
  `jax.vmap(step_jit)` inside a `jax.lax.scan` of length `rollout_steps`.
- Self-play: both players use the same `params`. Per env we randomly assign
  which player is the learner at reset time; PPO trains on the learner's
  decision rows only.
- Optional curriculum: train vs `versions/kaggle700_current_heuristic` until a
  rolling win-rate threshold is met, then continue self-play.
- Reward: pure terminal +1 / 0 / -1 from `state.rewards[learner_player]`.
  No shaping.
- GAE: gamma = 0.9999, lambda = 0.95 by default.
- Optimizer: optax cosine-decayed LR with `clip_by_global_norm`.

The trainer is structured so the rollout and the PPO update each live in a
single jit. Comet spawning happens host-side only when the env step reaches
one of `COMET_SPAWN_STEPS`; otherwise the entire rollout stays on device.
"""

from __future__ import annotations

import argparse
import functools
import math
import random
import time
from dataclasses import dataclass
from pathlib import Path

import flax.serialization
import jax
import jax.numpy as jnp
import jax.tree_util as tu
import numpy as np
import optax

from orbit_wars import (
    BUCKET_COUNT,
    COMET_SPAWN_STEPS,
    FLEET_FEATURE_DIM,
    GLOBAL_FEATURE_DIM,
    MAX_FLEETS,
    MAX_MOVES_PER_PLAYER,
    MAX_PLANETS,
    PLANET_FEATURE_DIM,
    batched_step,
    compose_action_grid,
    encode_observation,
    reset,
)
from orbit_wars.decode import INTERCEPT_ITERATIONS
from orbit_wars.heuristic_opponent import batched_heuristic_actions, load_heuristic_agent
from orbit_wars.rollout import pack_padded_actions, sample_actions
from orbit_wars.state import OrbitWarsState
from orbit_wars.step import _maybe_spawn_comet_numpy
from policy import PlanetPolicy
from ppo import compute_gae, explained_variance, ppo_loss_fn


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------


@dataclass(slots=True)
class TrainConfig:
    seed: int = 0
    run_name: str = "jax_ppo_transformer"
    save_dir: str = "artifacts"

    # Env / batch
    num_envs: int = 16
    episode_steps: int = 200
    rollout_steps: int = 32
    intercept_iterations: int = INTERCEPT_ITERATIONS
    enable_planet_block: bool = True
    enable_incoming_projection: bool = True

    # Model
    d_model: int = 96
    num_heads: int = 4
    num_layers: int = 3
    bucket_count: int = BUCKET_COUNT

    # PPO
    total_updates: int = 200
    epochs: int = 3
    minibatch_size: int = 1024
    gamma: float = 0.9999
    gae_lambda: float = 0.95
    clip_coef: float = 0.2
    ent_coef: float = 0.01
    vf_coef: float = 0.5
    lr_start: float = 1e-3
    lr_end: float = 1e-5
    max_grad_norm: float = 0.5
    weight_decay: float = 0.0

    # Logging / checkpoint
    log_every: int = 1
    checkpoint_every: int = 50

    # Opponent / curriculum
    opponent: str = "selfplay"  # selfplay | heuristic | curriculum
    heuristic_win_rate: float = 0.35
    heuristic_window_episodes: int = 80
    heuristic_path: str | None = None


def load_config(path: str | Path) -> TrainConfig:
    import yaml

    data = yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}
    env = data.get("env", {})
    model = data.get("model", {})
    ppo = data.get("ppo", {})
    training = data.get("training", {})
    heur_path = training.get("heuristic_path")
    return TrainConfig(
        seed=int(data.get("seed", 0)),
        run_name=str(data.get("run_name", "jax_ppo_transformer")),
        save_dir=str(data.get("save_dir", "artifacts")),
        num_envs=int(env.get("num_envs", 16)),
        episode_steps=int(env.get("episode_steps", 200)),
        rollout_steps=int(env.get("rollout_steps", 32)),
        intercept_iterations=int(env.get("intercept_iterations", INTERCEPT_ITERATIONS)),
        enable_planet_block=bool(env.get("enable_planet_block", True)),
        enable_incoming_projection=bool(env.get("enable_incoming_projection", True)),
        d_model=int(model.get("d_model", 96)),
        num_heads=int(model.get("num_heads", 4)),
        num_layers=int(model.get("num_layers", 3)),
        bucket_count=int(model.get("bucket_count", BUCKET_COUNT)),
        total_updates=int(ppo.get("total_updates", 200)),
        epochs=int(ppo.get("epochs", 3)),
        minibatch_size=int(ppo.get("minibatch_size", 1024)),
        gamma=float(ppo.get("gamma", 0.9999)),
        gae_lambda=float(ppo.get("gae_lambda", 0.95)),
        clip_coef=float(ppo.get("clip_coef", 0.2)),
        ent_coef=float(ppo.get("ent_coef", 0.01)),
        vf_coef=float(ppo.get("vf_coef", 0.5)),
        lr_start=float(ppo.get("lr_start", 1e-3)),
        lr_end=float(ppo.get("lr_end", 1e-5)),
        max_grad_norm=float(ppo.get("max_grad_norm", 0.5)),
        weight_decay=float(ppo.get("weight_decay", 0.0)),
        log_every=int(data.get("log_every", 1)),
        checkpoint_every=int(data.get("checkpoint_every", 50)),
        opponent=str(training.get("opponent", "selfplay")),
        heuristic_win_rate=float(training.get("heuristic_win_rate", 0.35)),
        heuristic_window_episodes=int(training.get("heuristic_window_episodes", 80)),
        heuristic_path=None if heur_path in (None, "null") else str(heur_path),
    )


# ---------------------------------------------------------------------------
# Env management (host-side comet spawn + on-device step)
# ---------------------------------------------------------------------------


def make_initial_states(cfg: TrainConfig, seed_base: int) -> tuple[OrbitWarsState, np.ndarray]:
    """Build initial batched states and per-env learner-player assignment."""
    rng = random.Random(seed_base)
    states = []
    learner_players = np.zeros(cfg.num_envs, dtype=np.int32)
    for i in range(cfg.num_envs):
        states.append(reset(seed_base + i, episode_steps=cfg.episode_steps))
        learner_players[i] = rng.randint(0, 1)
    batched = tu.tree_map(lambda *xs: jnp.stack(xs), *states)
    return batched, learner_players


def maybe_spawn_comets_host(batched_states: OrbitWarsState, cfg: TrainConfig) -> OrbitWarsState:
    """Run host-side comet spawn on each env only when the env's next step is
    a comet-spawn step. Cheap: 0 envs touch most updates."""
    next_step = int(np.asarray(batched_states.step)[0]) + 1
    if next_step not in COMET_SPAWN_STEPS:
        return batched_states
    new_states = []
    n = int(batched_states.step.shape[0])
    for i in range(n):
        single = tu.tree_map(lambda x, i=i: x[i], batched_states)
        new_states.append(_maybe_spawn_comet_numpy(single))
    return tu.tree_map(lambda *xs: jnp.stack(xs), *new_states)


# ---------------------------------------------------------------------------
# Rollout (one rollout-step batched across envs)
# ---------------------------------------------------------------------------


def policy_apply_factory(model: PlanetPolicy):
    def apply_fn(params, **kwargs):
        return model.apply(params, **kwargs)

    return jax.jit(apply_fn)


def _gather_by_player(zero_t, one_t, learner_players: jnp.ndarray):
    """Select per-env rows from player-0 or player-1 tensors."""
    lp = learner_players.astype(jnp.bool_)
    if zero_t.ndim == 1:
        return jnp.where(lp, one_t, zero_t)
    if zero_t.ndim == 2:
        return jnp.where(lp[:, None], one_t, zero_t)
    if zero_t.ndim == 3:
        return jnp.where(lp[:, None, None], one_t, zero_t)
    return jnp.where(lp[:, None, None, None], one_t, zero_t)


def sample_both_players_factory(model: PlanetPolicy, grid_fn):
    """Sample policy actions for both seats. Uses params for learner, opp_params for opponent."""

    @jax.jit
    def sample(states: OrbitWarsState, params, opp_params, rng, learner_players):
        rng, k0, k1 = jax.random.split(rng, 3)
        
        feats0 = jax.vmap(encode_observation, in_axes=(0, None))(states, jnp.int32(0))
        feats1 = jax.vmap(encode_observation, in_axes=(0, None))(states, jnp.int32(1))
        
        def _gather_feats(f0, f1, is_p0):
            # Expands the (B,) mask to the shape of f0/f1 for jnp.where
            mask = is_p0
            for _ in range(f0.ndim - 1):
                mask = mask[..., None]
            return jnp.where(mask, f0, f1)

        is_learner_p0 = (learner_players == 0)
        is_opp_p0 = (learner_players == 1)

        feats_learner = jax.tree_util.tree_map(lambda f0, f1: _gather_feats(f0, f1, is_learner_p0), feats0, feats1)
        feats_opp = jax.tree_util.tree_map(lambda f0, f1: _gather_feats(f0, f1, is_opp_p0), feats0, feats1)

        out_learner = model.apply(params, **feats_learner)
        out_opp = model.apply(opp_params, **feats_opp)

        out0 = jax.tree_util.tree_map(lambda l, o: _gather_feats(l, o, is_learner_p0), out_learner, out_opp)
        out1 = jax.tree_util.tree_map(lambda l, o: _gather_feats(l, o, is_opp_p0), out_learner, out_opp)

        grid0 = jax.vmap(grid_fn, in_axes=(0, None))(states, jnp.int32(0))
        s0 = sample_actions(k0, out0.target_logits, out0.bucket_logits, grid0)
        a0, m0, em0 = pack_padded_actions(s0["target_idx"], s0["bucket_idx"], s0["source_valid"], grid0)

        grid1 = jax.vmap(grid_fn, in_axes=(0, None))(states, jnp.int32(1))
        s1 = sample_actions(k1, out1.target_logits, out1.bucket_logits, grid1)
        a1, m1, em1 = pack_padded_actions(s1["target_idx"], s1["bucket_idx"], s1["source_valid"], grid1)
        
        return (a0, m0, a1, m1, s0, s1, out0, out1, grid0, grid1, feats0, feats1, em0, em1, rng)

    return sample


def sample_learner_factory(model: PlanetPolicy, grid_fn):
    """Sample policy actions for the learner seat only (player varies per env)."""

    @jax.jit
    def sample(states: OrbitWarsState, params, rng, learner_players):
        rng, k0 = jax.random.split(rng)
        feats = jax.vmap(encode_observation, in_axes=(0, 0))(states, learner_players)
        out = model.apply(params, **feats)
        grid = jax.vmap(grid_fn, in_axes=(0, 0))(states, learner_players)
        sampled = sample_actions(k0, out.target_logits, out.bucket_logits, grid)
        actions, mask, executed_mask = pack_padded_actions(
            sampled["target_idx"], sampled["bucket_idx"], sampled["source_valid"], grid
        )
        return actions, mask, executed_mask, sampled, out, grid, feats, rng

    return sample


def learner_record_from_samples(
    learner_players: jnp.ndarray,
    s0,
    s1,
    out0,
    out1,
    grid0,
    grid1,
    feats0,
    feats1,
    em0,
    em1,
    new_states: OrbitWarsState,
) -> dict:
    learner_feats = jax.tree_util.tree_map(
        lambda z, o: _gather_by_player(z, o, learner_players), feats0, feats1,
    )
    learner_value = _gather_by_player(out0.value, out1.value, learner_players)
    target_idx = _gather_by_player(s0["target_idx"], s1["target_idx"], learner_players)
    bucket_idx = _gather_by_player(s0["bucket_idx"], s1["bucket_idx"], learner_players)
    log_prob = _gather_by_player(s0["log_prob"], s1["log_prob"], learner_players)
    executed_mask = _gather_by_player(em0, em1, learner_players)
    target_has_bucket = _gather_by_player(
        jnp.any(grid0["full_valid"], axis=-1),
        jnp.any(grid1["full_valid"], axis=-1),
        learner_players,
    )
    bucket_valid = _gather_by_player(grid0["full_valid"], grid1["full_valid"], learner_players)

    reward = jnp.where(
        new_states.done & (learner_players == 0),
        new_states.rewards[:, 0],
        jnp.where(
            new_states.done & (learner_players == 1),
            new_states.rewards[:, 1],
            jnp.zeros_like(new_states.rewards[:, 0]),
        ),
    )
    
    # Combined reward for GAE: terminal + shaping
    step_reward_raw = _gather_by_player(new_states.step_rewards[:, 0], new_states.step_rewards[:, 1], learner_players)
    total_reward = reward + step_reward_raw

    opp_reward = jnp.where(
        new_states.done & (learner_players == 0),
        new_states.rewards[:, 1],
        jnp.where(
            new_states.done & (learner_players == 1),
            new_states.rewards[:, 0],
            jnp.zeros_like(new_states.rewards[:, 1]),
        ),
    )
    return {
        "planet_features": learner_feats["planet_features"],
        "planet_mask": learner_feats["planet_mask"],
        "fleet_features": learner_feats["fleet_features"],
        "fleet_mask": learner_feats["fleet_mask"],
        "global_features": learner_feats["global_features"],

        "target_idx": target_idx,
        "bucket_idx": bucket_idx,
        "log_prob": log_prob,
        "executed_mask": executed_mask,
        "target_has_bucket": target_has_bucket,
        "bucket_valid": bucket_valid,
        "value": learner_value,
        "reward": total_reward,
        "terminal_reward": reward,
        "opp_reward": opp_reward,
        "done": new_states.done,
    }


def learner_record_from_single(
    learner_players: jnp.ndarray,
    sampled,
    out,
    grid,
    feats,
    executed_mask,
    new_states: OrbitWarsState,
) -> dict:
    target_has_bucket = jnp.any(grid["full_valid"], axis=-1)
    bucket_valid = grid["full_valid"]

    batch = jnp.arange(new_states.rewards.shape[0])
    lp = learner_players.astype(jnp.int32)
    opp = (1 - lp).astype(jnp.int32)
    reward = new_states.rewards[batch, lp]
    opp_reward = new_states.rewards[batch, opp]

    reward = jnp.where(new_states.done, reward, jnp.zeros_like(reward))
    opp_reward = jnp.where(new_states.done, opp_reward, jnp.zeros_like(opp_reward))
    
    step_reward_raw = new_states.step_rewards[batch, lp]
    total_reward = reward + step_reward_raw

    return {
        "planet_features": feats["planet_features"],
        "planet_mask": feats["planet_mask"],
        "fleet_features": feats["fleet_features"],
        "fleet_mask": feats["fleet_mask"],
        "global_features": feats["global_features"],
        "target_idx": sampled["target_idx"],
        "bucket_idx": sampled["bucket_idx"],
        "log_prob": sampled["log_prob"],
        "executed_mask": executed_mask,
        "target_has_bucket": target_has_bucket,
        "bucket_valid": bucket_valid,
        "value": out.value,
        "reward": total_reward,
        "terminal_reward": reward,
        "opp_reward": opp_reward,
        "done": new_states.done,
    }


def rollout_step_selfplay_factory(model: PlanetPolicy, grid_fn):
    sample = sample_both_players_factory(model, grid_fn)
    step_jit = __import__("orbit_wars.step", fromlist=["step_jit"]).step_jit

    @jax.jit
    def step_one(states: OrbitWarsState, params, opp_params, rng, learner_players, reset_pool):
        rng, k_sample, k_pool, k_lp = jax.random.split(rng, 4)
        a0, m0, a1, m1, s0, s1, out0, out1, grid0, grid1, feats0, feats1, em0, em1, k_sample = sample(
            states, params, opp_params, k_sample, learner_players
        )
        new_states = jax.vmap(step_jit)(states, a0, a1, m0, m1)
        record = learner_record_from_samples(
            learner_players, s0, s1, out0, out1, grid0, grid1, feats0, feats1, em0, em1, new_states,
        )
        
        dones = new_states.done
        
        # Auto-reset
        pool_size = reset_pool.step.shape[0]
        pool_indices = jax.random.randint(k_pool, (states.step.shape[0],), 0, pool_size)
        fresh_states = jax.tree_util.tree_map(lambda p: p[pool_indices], reset_pool)
        
        next_states = jax.tree_util.tree_map(
            lambda new_s, fresh_s: jnp.where(
                dones[(...,) + (None,) * (new_s.ndim - 1)], fresh_s, new_s
            ),
            new_states, fresh_states
        )
        
        new_learner_players = jax.random.randint(k_lp, (states.step.shape[0],), 0, 2)
        next_learner_players = jnp.where(dones, new_learner_players, learner_players)
        
        return next_states, record, rng, next_learner_players

    return step_one


def rollout_step_vs_heuristic_factory(model: PlanetPolicy, grid_fn):
    """Policy learner + frozen heuristic opponent (host-side opponent actions)."""
    sample = sample_learner_factory(model, grid_fn)
    step_jit = __import__("orbit_wars.step", fromlist=["step_jit"]).step_jit

    def step_one(
        states: OrbitWarsState,
        params,
        rng,
        learner_players,
        opponent_players_np: np.ndarray,
        heuristic_agent,
    ):
        actions, mask, executed_mask, sampled, out, grid, feats, rng = sample(states, params, rng, learner_players)
        ha0, hm0, ha1, hm1 = batched_heuristic_actions(states, opponent_players_np, heuristic_agent)

        is_learner_p0 = (learner_players == 0)
        final_a0 = jnp.where(is_learner_p0[:, None, None], actions, ha0)
        final_a1 = jnp.where(is_learner_p0[:, None, None], ha1, actions)
        final_m0 = jnp.where(is_learner_p0[:, None], mask, hm0)
        final_m1 = jnp.where(is_learner_p0[:, None], hm1, mask)
new_states = jax.vmap(step_jit)(states, final_a0, final_a1, final_m0, final_m1)
record = learner_record_from_single(
    learner_players, sampled, out, grid, feats, executed_mask, new_states,
)
return new_states, record, rng
    return step_one


def reset_done_envs(states: OrbitWarsState, dones_np: np.ndarray, next_seed: int, cfg: TrainConfig) -> tuple[OrbitWarsState, int, np.ndarray]:
    """Host-side resets for envs whose episodes ended. Returns (new_states,
    next_seed, new_learner_players_for_those_envs)."""
    n = int(states.step.shape[0])
    if not dones_np.any():
        return states, next_seed, np.zeros(n, dtype=np.int32)

    rng = random.Random(next_seed)
    refreshed = []
    new_lp = np.zeros(n, dtype=np.int32)
    states_list = [tu.tree_map(lambda x, i=i: x[i], states) for i in range(n)]
    for i in range(n):
        if dones_np[i]:
            s = reset(next_seed, episode_steps=cfg.episode_steps)
            new_lp[i] = rng.randint(0, 1)
            next_seed += 1
            refreshed.append(s)
        else:
            refreshed.append(states_list[i])
    new_states = tu.tree_map(lambda *xs: jnp.stack(xs), *refreshed)
    return new_states, next_seed, new_lp


# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------


def make_optimizer(cfg: TrainConfig):
    schedule = optax.cosine_decay_schedule(
        init_value=cfg.lr_start,
        decay_steps=cfg.total_updates,
        alpha=cfg.lr_end / max(cfg.lr_start, 1e-12),
    )
    return optax.chain(optax.clip_by_global_norm(cfg.max_grad_norm), optax.adamw(schedule, weight_decay=cfg.weight_decay)), schedule


def init_policy_params(rng, model: PlanetPolicy):
    example = {
        "planet_features": jnp.zeros((1, MAX_PLANETS, PLANET_FEATURE_DIM), jnp.float32),
        "planet_mask": jnp.ones((1, MAX_PLANETS), jnp.bool_),
        "fleet_features": jnp.zeros((1, MAX_FLEETS, FLEET_FEATURE_DIM), jnp.float32),
        "fleet_mask": jnp.ones((1, MAX_FLEETS), jnp.bool_),
        "global_features": jnp.zeros((1, GLOBAL_FEATURE_DIM), jnp.float32),
    }
    return model.init(rng, **example), example


def make_update_step(model: PlanetPolicy, optimizer, cfg: TrainConfig):
    @jax.jit
    def update(params, opt_state, batch):
        def loss(p):
            return ppo_loss_fn(p, model.apply, batch, cfg.clip_coef, cfg.vf_coef, cfg.ent_coef)

        (l, metrics), grads = jax.value_and_grad(loss, has_aux=True)(params)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, l, metrics

    return update


def train(cfg: TrainConfig) -> None:
    rng = jax.random.PRNGKey(cfg.seed)
    rng, init_rng = jax.random.split(rng)

    model = PlanetPolicy(
        planet_count=MAX_PLANETS, fleet_count=MAX_FLEETS,
        d_model=cfg.d_model, num_heads=cfg.num_heads, num_layers=cfg.num_layers,
        bucket_count=cfg.bucket_count,
    )
    params, _ = init_policy_params(init_rng, model)
    opp_params = params
    optimizer, _ = make_optimizer(cfg)
    opt_state = optimizer.init(params)

    grid_fn = functools.partial(
        compose_action_grid,
        intercept_iterations=cfg.intercept_iterations,
        enable_planet_block=cfg.enable_planet_block,
        enable_incoming_projection=cfg.enable_incoming_projection,
    )
    rollout_selfplay = rollout_step_selfplay_factory(model, grid_fn)
    rollout_vs_heuristic = rollout_step_vs_heuristic_factory(model, grid_fn)
    update_step = make_update_step(model, optimizer, cfg)

    opponent_mode = cfg.opponent.lower()
    if opponent_mode not in ("selfplay", "heuristic", "curriculum"):
        raise ValueError(f"unknown opponent mode: {cfg.opponent}")
    active_mode = "heuristic" if opponent_mode in ("heuristic", "curriculum") else "selfplay"
    curriculum_switched = False

    save_dir = Path(cfg.save_dir) / cfg.run_name
    save_dir.mkdir(parents=True, exist_ok=True)
    
    log_file_path = save_dir / "training.log"
    log_file = log_file_path.open("a", encoding="utf-8")

    def log_print(msg: str) -> None:
        print(msg, flush=True)
        log_file.write(msg + "\n")
        log_file.flush()

    heuristic_agent = None
    if active_mode == "heuristic":
        heur_path = Path(cfg.heuristic_path) if cfg.heuristic_path else None
        heuristic_agent = load_heuristic_agent(heur_path)
        log_print(f"Heuristic opponent loaded from {heur_path or 'default'}")

    # Pre-generate reset pool for auto-reset
    pool_size = max(256, cfg.num_envs * 4)
    log_print(f"Generating reset pool of size {pool_size}...")
    reset_pool_states, _ = make_initial_states(cfg, cfg.seed + 100000)
    # We need a pool of `pool_size`. make_initial_states uses `num_envs` so we just call it with a custom config.
    import dataclasses
    cfg_pool = dataclasses.replace(cfg, num_envs=pool_size)
    reset_pool, _ = make_initial_states(cfg_pool, cfg.seed + 100000)
    
    seed_base = cfg.seed * 10000 + 1
    states, learner_players_np = make_initial_states(cfg, seed_base)
    learner_players = jnp.asarray(learner_players_np)
    next_seed = seed_base + cfg.num_envs

    log_print(
        f"JAX devices: {jax.devices()} | envs={cfg.num_envs} rollout={cfg.rollout_steps} "
        f"updates={cfg.total_updates} opponent={opponent_mode} active={active_mode}"
    )
    log_print(
        "update |     mode | lrnr_wr | W-L-D | episodes | mean_ret | env_sps | "
        "  loss | pol_loss | val_loss | entropy |     ev | approx_kl | clip_fr"
    )

    t_start = time.perf_counter()
    total_env_steps = 0
    finished_returns_window: list[float] = []
    heuristic_returns_window: list[float] = []
    learner_wins = learner_losses = learner_draws = 0

    for update_idx in range(1, cfg.total_updates + 1):
        t_rollout = time.perf_counter()
        rollout_records = []
        for _ in range(cfg.rollout_steps):
            if active_mode != "selfplay":
                states = maybe_spawn_comets_host(states, cfg)
            rng, sub = jax.random.split(rng)
            if active_mode == "selfplay":
                states, rec, rng, learner_players = rollout_selfplay(states, params, opp_params, sub, learner_players, reset_pool)
            else:
                opp_np = 1 - learner_players_np
                states, rec, rng = rollout_vs_heuristic(
                    states, params, sub, learner_players, opp_np, heuristic_agent,
                )
            rollout_records.append(rec)
            
            if active_mode != "selfplay":
                done_np = np.asarray(rec["done"])
                if done_np.any():
                    reward_np = np.asarray(rec["reward"])
                    finished_returns_window.extend(reward_np[done_np].tolist())
                    opp_reward_np = np.asarray(rec.get("opp_reward", np.zeros_like(reward_np)))
                    heuristic_returns_window.extend(reward_np[done_np].tolist())
                    wins = np.sum((reward_np > opp_reward_np) & done_np)
                    losses = np.sum((reward_np < opp_reward_np) & done_np)
                    draws = np.sum((reward_np == opp_reward_np) & done_np)
                    learner_wins += int(wins)
                    learner_losses += int(losses)
                    learner_draws += int(draws)
    
                    states, next_seed, new_lp = reset_done_envs(states, done_np, next_seed, cfg)
                    learner_players_np = np.where(done_np, new_lp, learner_players_np)
                    learner_players = jnp.asarray(learner_players_np)

        # For selfplay, process dones after the rollout loop to avoid blocking GPU
        if active_mode == "selfplay":
            # Just extract the data once it's all done
            dones_batch = jnp.stack([r["done"] for r in rollout_records], axis=1)
            rewards_batch = jnp.stack([r["reward"] for r in rollout_records], axis=1)
            opp_rewards_batch = jnp.stack([r["opp_reward"] for r in rollout_records], axis=1)
            done_mask = np.asarray(dones_batch)
            reward_vals = np.asarray(rewards_batch)
            opp_reward_vals = np.asarray(opp_rewards_batch)
            if done_mask.any():
                finished_returns_window.extend(reward_vals[done_mask].tolist())
                heuristic_returns_window.extend(reward_vals[done_mask].tolist())
                wins = np.sum((reward_vals > opp_reward_vals) & done_mask)
                losses = np.sum((reward_vals < opp_reward_vals) & done_mask)
                draws = np.sum((reward_vals == opp_reward_vals) & done_mask)
                learner_wins += int(wins)
                learner_losses += int(losses)
                learner_draws += int(draws)

        rollout_s = time.perf_counter() - t_rollout
        total_env_steps += cfg.rollout_steps * cfg.num_envs

        # ------- bootstrap value for GAE -------
        feats_boot = jax.vmap(encode_observation, in_axes=(0, 0))(states, learner_players)
        out_boot = model.apply(params, **feats_boot)
        next_value = out_boot.value                                   # (B,)

        # ------- assemble (B, T) tensors -------
        # rollout_records is a list of dicts; stack along T axis.
        def stack_t(key, leaves):
            return jnp.stack([r[key] for r in leaves], axis=1)        # (B, T, ...)

        rewards = stack_t("reward", rollout_records)                   # (B, T)
        dones = stack_t("done", rollout_records)                       # (B, T)
        values = stack_t("value", rollout_records)                     # (B, T)

        adv, ret = compute_gae(rewards, values, dones, next_value, cfg.gamma, cfg.gae_lambda)
        # Normalize advantages.
        adv_mean = jnp.mean(adv)
        adv_std = jnp.std(adv) + 1e-8
        adv = (adv - adv_mean) / adv_std

        # Flatten to (N = B*T, ...).
        def flatten(arr):
            shape = arr.shape
            return arr.reshape((shape[0] * shape[1],) + shape[2:])

        flat = {}
        for k in (
            "planet_features", "planet_mask",
            "fleet_features", "fleet_mask",
            "global_features",
            "target_idx", "bucket_idx", "log_prob", "executed_mask",
            "target_has_bucket", "bucket_valid",
        ):
            flat[k] = flatten(stack_t(k, rollout_records))
        flat["old_log_prob"] = flat.pop("log_prob")
        flat["advantages"] = flatten(adv)
        flat["returns"] = flatten(ret)

        n_rows = flat["advantages"].shape[0]

        # ------- PPO update -------
        t_train = time.perf_counter()
        metrics_accum = {
            "loss": 0.0, "policy_loss": 0.0, "value_loss": 0.0,
            "entropy": 0.0, "approx_kl": 0.0, "clip_fraction": 0.0,
        }
        opt_steps = 0
        for _ in range(cfg.epochs):
            perm = np.random.permutation(n_rows)
            for start in range(0, n_rows, cfg.minibatch_size):
                idx = perm[start : start + cfg.minibatch_size]
                mb = {k: v[idx] for k, v in flat.items()}
                params, opt_state, loss_val, m = update_step(params, opt_state, mb)
                metrics_accum["loss"] += float(loss_val)
                for k in ("policy_loss", "value_loss", "entropy", "approx_kl", "clip_fraction"):
                    metrics_accum[k] += float(m[k])
                opt_steps += 1

        train_s = time.perf_counter() - t_train
        if opt_steps:
            for k in metrics_accum:
                metrics_accum[k] /= opt_steps

        ev = float(explained_variance(flat["returns"], flat["advantages"] + flat["returns"] - flat["advantages"]))  # ≈ EV(returns, returns) sanity
        # Better: compute new values on (subset of) batch — quick approximation:
        idx = np.random.choice(n_rows, size=min(1024, n_rows), replace=False)
        sub = {k: v[idx] for k, v in flat.items()}
        v_sub = model.apply(
            params,
            planet_features=sub["planet_features"], planet_mask=sub["planet_mask"],
            fleet_features=sub["fleet_features"], fleet_mask=sub["fleet_mask"],
            global_features=sub["global_features"],
        ).value
        ev = float(explained_variance(sub["returns"], v_sub))

        elapsed = time.perf_counter() - t_start
        env_sps = total_env_steps / elapsed

        mean_ret = float(np.mean(finished_returns_window[-50:])) if finished_returns_window else float("nan")
        episodes = len(finished_returns_window)

        window = heuristic_returns_window[-cfg.heuristic_window_episodes :]
        learner_wr = float(np.mean([1.0 if r > 0 else 0.0 for r in window])) if window else float("nan")
        wld = f"{learner_wins}-{learner_losses}-{learner_draws}"

        if update_idx % cfg.log_every == 0:
            log_print(
                f"{update_idx:6d} | {active_mode:8s} | "
                f"{learner_wr:7.1%} | "
                f"{wld:>5s} | "
                f"{episodes:7d} | {mean_ret:+.3f} | {env_sps:7.0f} | "
                f"{metrics_accum['loss']:.4f} | {metrics_accum['policy_loss']:+.4f} | "
                f"{metrics_accum['value_loss']:.4f} | {metrics_accum['entropy']:.3f} | "
                f"{ev:+.3f} | {metrics_accum['approx_kl']:.5f} | {metrics_accum['clip_fraction']:.3f}"
            )
            learner_wins = learner_losses = learner_draws = 0

        if active_mode == "selfplay" and len(window) >= cfg.heuristic_window_episodes and learner_wr >= 0.54:
            log_print(f"Update {update_idx}: Self-play winrate {learner_wr:.1%} >= 54.0%. Updating opponent parameters.")
            opp_params = params
            heuristic_returns_window.clear()

        if (
            opponent_mode == "curriculum"
            and active_mode == "heuristic"
            and not curriculum_switched
            and len(window) >= cfg.heuristic_window_episodes
            and learner_wr >= cfg.heuristic_win_rate
        ):
            active_mode = "selfplay"
            curriculum_switched = True
            log_print("=" * 72)
            log_print(
                f"CURRICULUM SWITCH at update {update_idx}: "
                f"heuristic win rate {learner_wr:.1%} >= {cfg.heuristic_win_rate:.1%}. "
                f"Continuing with self-play for remaining updates."
            )
            log_print("=" * 72)

        if update_idx % cfg.checkpoint_every == 0 or update_idx == cfg.total_updates:
            blob = np.frombuffer(flax.serialization.to_bytes(params), dtype=np.uint8)
            np.savez(save_dir / "ckpt_last.npz", update=update_idx, params=blob)
            np.savez(save_dir / f"ckpt_{update_idx:06d}.npz", update=update_idx, params=blob)

    total_elapsed = time.perf_counter() - t_start
    log_print(f"Done. total_env_steps={total_env_steps} elapsed={total_elapsed:.1f}s")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/smoke_transformer.yaml")
    args = parser.parse_args()
    train(load_config(args.config))


if __name__ == "__main__":
    main()


## Config


In [ ]:
%%writefile configs/transformer_selfplay.yaml
# Full self-play config — intended for Kaggle GPU.
# Locally on CPU this will be slow; use smoke_transformer.yaml for iteration.
seed: 0
run_name: jax_ppo_transformer
save_dir: artifacts

env:
  num_envs: 32           # 64 OOMs on T4 (16 GB). Bump back up on bigger GPUs.
  episode_steps: 500
  rollout_steps: 32

model:
  d_model: 96
  num_heads: 4
  num_layers: 3
  bucket_count: 8

ppo:
  total_updates: 5000
  epochs: 3
  minibatch_size: 256    # peak attention memory ~ B * H * T^2; 256 fits T4
  gamma: 0.9999
  gae_lambda: 0.95
  clip_coef: 0.2
  ent_coef: 0.01
  vf_coef: 0.5
  lr_start: 1.0e-3
  lr_end: 1.0e-5
  max_grad_norm: 0.5

log_every: 5
checkpoint_every: 100


## Start Training


In [ ]:
import os
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.85'

!python train_ppo.py --config configs/transformer_selfplay.yaml


## Export Submission


In [ ]:
%%writefile export_jax_submission.py
"""Package a trained JAX policy into a Kaggle-ready submission zip.

Usage (from `rl_training_jax/`):

    python scripts/export_jax_submission.py \
        --checkpoint artifacts/jax_ppo_transformer/ckpt_last.npz \
        --config configs/transformer_selfplay.yaml \
        --output ../submission_jax.zip

What it does:

1. Loads the checkpoint (`.npz` produced by `train_ppo.py`).
2. Re-serializes the flax params into `weights/policy.msgpack`.
3. Writes `weights/model_config.json` with d_model/n_heads/etc.
4. Copies a minimal subset of `orbit_wars/` + `policy.py` into
   `submission_jax/src/`.
5. Zips the `submission_jax/` directory.
"""

from __future__ import annotations

import argparse
import json
import shutil
import sys
import zipfile
from pathlib import Path

import flax.serialization
import jax
import jax.numpy as jnp
import numpy as np

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, ".")

from orbit_wars import MAX_FLEETS, MAX_PLANETS, PLANET_FEATURE_DIM
from policy import PlanetPolicy
from train_ppo import load_config


# These are the files actually needed at inference time. Training-only files
# (env, reset, step, comet, reference) are excluded to keep the submission lean
# and free of any kaggle_environments dependency.
INFERENCE_FILES = [
    "__init__.py",
    "constants.py",
    "state.py",
    "geometry.py",
    "convert.py",
    "features_jax.py",
    "decode.py",
]


def _filter_init_imports(text: str) -> str:
    """Strip imports of training-only modules from `orbit_wars/__init__.py`
    so the submission package doesn't need env/step/reset/comet/reference."""
    drop_lines = (
        "from .env import",
        "from .reference import",
        "from .reset import",
        "from .step import",
        "from .comet import",
    )
    out = []
    for line in text.splitlines(keepends=True):
        if any(line.startswith(d) for d in drop_lines):
            continue
        out.append(line)
    # Also remove these symbols from __all__.
    text = "".join(out)
    for sym in ("OrbitWarsJaxEnv", "VectorOrbitWarsEnv", "reset", "step", "step_jit",
                "batched_step", "reference_reset", "reference_step"):
        text = text.replace(f'    "{sym}",\n', "")
    return text


def export(checkpoint: Path, config: Path, output: Path) -> None:
    cfg = load_config(config)
    submission_dir = Path("submission_jax")
    src_dir = submission_dir / "src"
    weights_dir = submission_dir / "weights"
    pkg_dir = src_dir / "orbit_wars"

    # Clean & recreate.
    if pkg_dir.exists():
        shutil.rmtree(pkg_dir)
    pkg_dir.mkdir(parents=True, exist_ok=True)
    weights_dir.mkdir(parents=True, exist_ok=True)

    # Copy inference files.
    src_pkg = Path("orbit_wars")
    for fname in INFERENCE_FILES:
        text = (src_pkg / fname).read_text(encoding="utf-8")
        if fname == "__init__.py":
            text = _filter_init_imports(text)
        (pkg_dir / fname).write_text(text, encoding="utf-8")

    # Copy policy.py.
    shutil.copy2("policy.py", src_dir / "policy.py")

    # Load checkpoint and re-serialize params.
    if not checkpoint.exists():
        raise FileNotFoundError(f"checkpoint not found: {checkpoint}")
    ckpt = np.load(checkpoint, allow_pickle=False)
    blob = bytes(ckpt["params"].tobytes())

    # Sanity: round-trip through flax with the right model shape to validate.
    model = PlanetPolicy(
        planet_count=MAX_PLANETS, fleet_count=MAX_FLEETS,
        d_model=cfg.d_model, num_heads=cfg.num_heads,
        num_layers=cfg.num_layers, bucket_count=cfg.bucket_count,
    )
    example = {
        "planet_features": jnp.zeros((1, MAX_PLANETS, PLANET_FEATURE_DIM), jnp.float32),
        "planet_mask": jnp.ones((1, MAX_PLANETS), jnp.bool_),
    }
    init_params = model.init(jax.random.PRNGKey(0), **example)
    params = flax.serialization.from_bytes(init_params, blob)
    _ = model.apply(params, **example)            # forward smoke
    blob = flax.serialization.to_bytes(params)

    (weights_dir / "policy.msgpack").write_bytes(blob)
    (weights_dir / "model_config.json").write_text(
        json.dumps({
            "d_model": cfg.d_model,
            "num_heads": cfg.num_heads,
            "num_layers": cfg.num_layers,
            "bucket_count": cfg.bucket_count,
            "planet_feature_dim": PLANET_FEATURE_DIM,
        }, indent=2),
        encoding="utf-8",
    )

    # Build the zip.
    output.parent.mkdir(parents=True, exist_ok=True)
    if output.exists():
        output.unlink()
    with zipfile.ZipFile(output, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in submission_dir.rglob("*"):
            if path.is_file():
                arcname = path.relative_to(submission_dir.parent)
                zf.write(path, arcname=arcname)

    print(f"Wrote submission: {output}")
    print(f"Submission size: {output.stat().st_size / 1024:.1f} KiB")
    print(f"Weights bytes:   {len(blob)} ({len(blob)/1024:.1f} KiB)")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True, type=Path)
    parser.add_argument("--config", required=True, type=Path)
    parser.add_argument("--output", default="../submission_jax.zip", type=Path)
    args = parser.parse_args()
    export(args.checkpoint.resolve(), args.config.resolve(), args.output.resolve())


if __name__ == "__main__":
    main()


In [ ]:
!python export_jax_submission.py \
    --checkpoint artifacts/jax_ppo_transformer/ckpt_last.npz \
    --config configs/transformer_selfplay.yaml \
    --output submission_jax.zip
